# GA4 E-Commerce Funnel Analysis
**Dataset:** `bigquery-public-data.ga4_obfuscated_sample_ecommerce`  
**Stack:** BigQuery (SQL) → Pandas → Plotly  
**Date range:** Nov 2020 – Jan 2021 (full public dataset)

## What this notebook covers

| Section | Analysis |
|---|---|
| 1 | Setup & config |
| 2 | Schema exploration — event inventory, date range |
| 3 | **Session-scoped funnel** — conversion within a single visit |
| 4 | Revenue layer — $ impact at each funnel step |
| 5 | Device category breakdown |
| 5c | **Acquisition channel breakdown** — is the discovery loss a traffic problem? |
| 5e | **Data quality** — what `(not set)` sessions are; what `view_item` / `add_to_cart` record |
| 5f | **Statistical significance** — device and channel (Wilson intervals, chi-square) |
| 6 | Time-to-convert — median minutes between steps, plus a tracking check |
| 7 | Alternative entry: shoppers funnel starting at `view_item` |
| 7c | **Product category view → cart** — where to start the merchandising audit |
| 8 | Weekly cohort retention |
| 8c | **New vs returning sessions** — funnel and revenue per session by visitor type |
| 9 | Summary & hypotheses |
| 10 | **AI-powered insight generation** — LLM product brief |

> **Session vs user grain:** sections 3–7 use session-scoped funnels (each row = one visit).
> This matches how the GA4 UI reports conversion and gives sharper, more actionable numbers
> than user-scoped funnels where a past purchaser "carries over" into every future session.

## 1. Setup

In [1]:
# ── Section 1: Setup ──────────────────────────────────────────────────────────
# Suppress non-critical library warnings so notebook output stays readable.
import warnings
warnings.filterwarnings('ignore')

# Pandas: reshape query results into DataFrames for metrics and charts.
import pandas as pd
# Plotly: interactive funnel, bar, and heatmap visualizations.
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Project, dataset and date window come from config.py — shared with
# generate_dashboard.py so the two can never drift apart. Override without
# editing code, e.g. before launching Jupyter:
#   export GA4_PROJECT_ID="your-gcp-project"            # runs (and pays for) the queries
#   export GA4_DATASET="your-project.analytics_123456"  # default: public GA4 sample
#   export GA4_DATE_START="20240101"; export GA4_DATE_END="20240331"
# Widen the window carefully: each extra month adds ~$0.005 per query on the sample.
from config import (PROJECT_ID, DATASET, DATE_START, DATE_END, PERIOD_LABEL,
                    DATASET_LABEL, DATASET_MD, IS_PUBLIC_SAMPLE, make_client, describe)

# Client uses Application Default Credentials (gcloud auth application-default login).
client = make_client()
print(f"BigQuery client ready  |  {describe(client)}")

BigQuery client ready  |  dataset: bigquery-public-data.ga4_obfuscated_sample_ecommerce  |  window: 20201101 → 20210131 (Nov 2020 – Jan 2021)


## 2. Schema exploration
Confirm date range and event inventory before writing analysis queries.

In [2]:
# ── Section 2a: Dataset date range ───────────────────────────────────────────
# GA4 exports one table per day: events_YYYYMMDD. _TABLE_SUFFIX is the date suffix.
# Scan all daily tables to confirm earliest/latest dates and table count.
date_range_df = client.query(f"""
SELECT
  MIN(_TABLE_SUFFIX) AS earliest_date,
  MAX(_TABLE_SUFFIX) AS latest_date,
  COUNT(DISTINCT _TABLE_SUFFIX) AS num_daily_tables
FROM `{DATASET}.events_*`
""").to_dataframe()

print("Full dataset date range:")
display(date_range_df)


Full dataset date range:


,earliest_date,latest_date,num_daily_tables
0,20201101,20210131,92


In [3]:
# ── Section 2b: Event inventory ────────────────────────────────────────────
# List every event_name in the analysis window with volume and reach.
# Use this to validate funnel step events exist before building session logic.
events_df = client.query(f"""
SELECT
  event_name,
  COUNT(*)                       AS event_count,
  COUNT(DISTINCT user_pseudo_id) AS unique_users
FROM `{DATASET}.events_*`
WHERE _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
GROUP BY event_name
ORDER BY event_count DESC
""").to_dataframe()

print(f"Events in window {DATE_START}–{DATE_END}:")
display(events_df)


Events in window 20201101–20210131:


,event_name,event_count,unique_users
0,page_view,1350428,269792
1,user_engagement,1058721,213004
2,scroll,493072,138098
3,view_item,386068,61252
4,session_start,354970,267116
5,first_visit,257462,257314
6,view_promotion,190104,102443
7,add_to_cart,58543,12545
8,begin_checkout,38757,9715
9,select_item,31007,13180


## 3. Session-scoped funnel

**Grain:** one row = one session (`user_pseudo_id` + `ga_session_id`).
A session converts at step N only if it reached step N−1 first *within the same session*.

**Why session-scoped?**
- Matches GA4 UI funnel reports
- Avoids inflating conversion by counting past purchasers who return for unrelated visits
- Step-to-step rates reflect a single coherent user intent

**Funnel:** `session_start` → `view_item` → `add_to_cart` → `begin_checkout` → `purchase`

**What step 2 means here.** In this export `view_item` fires on listing, home and search pages
with a whole product list attached (Section 5e), so step 2 is labelled *View Products* — the
session **saw** products — not "opened a product page".

In [4]:
# ── Section 3: Session-scoped funnel (BigQuery) ─────────────────────────────
# Grain: one row per (user_pseudo_id, ga_session_id). Steps must occur in order
# within the same visit — mirrors GA4 UI funnel reporting.
session_funnel_sql = f"""
-- base: pull funnel events and extract ga_session_id from the event_params array.
WITH base AS (
  SELECT
    user_pseudo_id,
    (SELECT ep.value.int_value
     FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'ga_session_id'
     LIMIT 1)           AS session_id,
    event_name,
    event_timestamp
  FROM `{DATASET}.events_*`
  WHERE
    _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name IN ('session_start','view_item','add_to_cart','begin_checkout','purchase')
),

-- session_events: first occurrence of each event per session (dedupe repeats).
session_events AS (
  SELECT
    user_pseudo_id,
    session_id,
    event_name,
    MIN(event_timestamp) AS first_ts
  FROM base
  WHERE session_id IS NOT NULL
  GROUP BY user_pseudo_id, session_id, event_name
),

-- pivoted: one row per session; columns hold earliest timestamp per step.
pivoted AS (
  SELECT
    user_pseudo_id,
    session_id,
    MAX(CASE WHEN event_name = 'session_start'  THEN first_ts END) AS ts_session,
    MAX(CASE WHEN event_name = 'view_item'      THEN first_ts END) AS ts_view,
    MAX(CASE WHEN event_name = 'add_to_cart'    THEN first_ts END) AS ts_cart,
    MAX(CASE WHEN event_name = 'begin_checkout' THEN first_ts END) AS ts_checkout,
    MAX(CASE WHEN event_name = 'purchase'       THEN first_ts END) AS ts_purchase
  FROM session_events
  GROUP BY user_pseudo_id, session_id
),

-- sequential: count sessions reaching each step only if prior steps happened first.
sequential AS (
  SELECT
    COUNT(*)                                                                         AS step1_sessions,
    COUNTIF(ts_view     IS NOT NULL AND ts_view     >= ts_session)                  AS step2_view_item,
    COUNTIF(ts_cart     IS NOT NULL AND ts_cart     >= ts_view
                                   AND ts_view      >= ts_session)                  AS step3_add_to_cart,
    COUNTIF(ts_checkout IS NOT NULL AND ts_checkout >= ts_cart
                                   AND ts_cart      >= ts_view
                                   AND ts_view      >= ts_session)                  AS step4_begin_checkout,
    COUNTIF(ts_purchase IS NOT NULL AND ts_purchase >= ts_checkout
                                   AND ts_checkout  >= ts_cart
                                   AND ts_cart      >= ts_view
                                   AND ts_view      >= ts_session)                  AS step5_purchase
  FROM pivoted
  WHERE ts_session IS NOT NULL
)

SELECT * FROM sequential
"""

funnel_raw = client.query(session_funnel_sql).to_dataframe()
print("Session-scoped funnel counts:")
display(funnel_raw)


Session-scoped funnel counts:


,step1_sessions,step2_view_item,step3_add_to_cart,step4_begin_checkout,step5_purchase
0,354857,75261,14870,5205,2721


In [5]:
# ── Section 3b: Funnel metrics + Plotly visualization ─────────────────────
# Human-readable step labels aligned with GA4 funnel stage names.
step_labels = [
    "1. Session Start",
    "2. View Products",
    "3. Add to Cart",
    "4. Begin Checkout",
    "5. Purchase",
]
# Column names returned by the sequential CTE in session_funnel_sql.
step_cols = [
    "step1_sessions", "step2_view_item", "step3_add_to_cart",
    "step4_begin_checkout", "step5_purchase",
]

# Build a tidy funnel table for display and downstream LLM context (Section 10).
funnel_df = pd.DataFrame({
    "step":  step_labels,
    "sessions": [int(funnel_raw[c].iloc[0]) for c in step_cols],
})
# step_cvr: conversion from the previous step (% who advanced one stage).
funnel_df["step_cvr"]    = (funnel_df["sessions"] / funnel_df["sessions"].shift(1) * 100).round(1)
# overall_cvr: conversion from session start (% still in funnel at this step).
funnel_df["overall_cvr"] = (funnel_df["sessions"] / funnel_df["sessions"].iloc[0] * 100).round(1)
# drop_off: 100 - step_cvr — share lost between adjacent steps.
funnel_df["drop_off"]    = (100 - funnel_df["step_cvr"]).round(1)

print("Session funnel with conversion rates:")
display(funnel_df)

# Funnel chart: width of each stage = remaining sessions.
fig = px.funnel(
    funnel_df, x="sessions", y="step",
    title=f"GA4 Session-Scoped Funnel  ({DATE_START[:6]} – {DATE_END[:6]})",
    color_discrete_sequence=["#4C78A8"],
)
# Annotate each transition with drop-off % and step conversion (skip top of funnel).
for i, row in funnel_df.iterrows():
    if i == 0:
        continue
    fig.add_annotation(
        x=row["sessions"], y=row["step"],
        text=f"  ↓ {row['drop_off']}% drop-off  ({row['step_cvr']}% pass)",
        showarrow=False, xanchor="left",
        font=dict(size=11, color="#d62728"),
    )
fig.update_layout(height=450, margin=dict(l=160, r=260, t=60, b=40))
fig.show()


Session funnel with conversion rates:


,step,sessions,step_cvr,overall_cvr,drop_off
0,1. Session Start,354857,NaN,100.0,NaN
1,2. View Products,75261,21.2,21.2,78.8
2,3. Add to Cart,14870,19.8,4.2,80.2
3,4. Begin Checkout,5205,35.0,1.5,65.0
4,5. Purchase,2721,52.3,0.8,47.7


## 4. Revenue layer
Add `ecommerce.purchase_revenue` to see the **dollar impact** at the purchase step.
Also computes revenue per session entering the funnel and revenue per converting session.

In [6]:
# ── Section 4: Revenue layer ────────────────────────────────────────────────
# Restrict to sessions that completed the full sequential funnel, then sum
# ecommerce.purchase_revenue (only populated on purchase events).
revenue_sql = f"""
WITH base AS (
  SELECT
    user_pseudo_id,
    (SELECT ep.value.int_value FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'ga_session_id' LIMIT 1) AS session_id,
    event_name,
    event_timestamp,
    -- Revenue only present on purchase events
    ecommerce.purchase_revenue AS purchase_revenue
  FROM `{DATASET}.events_*`
  WHERE
    _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name IN ('session_start','view_item','add_to_cart','begin_checkout','purchase')
),
session_events AS (
  SELECT
    user_pseudo_id, session_id, event_name,
    MIN(event_timestamp)   AS first_ts,
    SUM(purchase_revenue)  AS session_revenue
  FROM base
  WHERE session_id IS NOT NULL
  GROUP BY user_pseudo_id, session_id, event_name
),
pivoted AS (
  SELECT
    user_pseudo_id, session_id,
    MAX(CASE WHEN event_name = 'session_start'  THEN first_ts END)      AS ts_session,
    MAX(CASE WHEN event_name = 'view_item'      THEN first_ts END)      AS ts_view,
    MAX(CASE WHEN event_name = 'add_to_cart'    THEN first_ts END)      AS ts_cart,
    MAX(CASE WHEN event_name = 'begin_checkout' THEN first_ts END)      AS ts_checkout,
    MAX(CASE WHEN event_name = 'purchase'       THEN first_ts END)      AS ts_purchase,
    MAX(CASE WHEN event_name = 'purchase'       THEN session_revenue END) AS revenue
  FROM session_events
  GROUP BY user_pseudo_id, session_id
),
-- converting_sessions: same sequential rules as Section 3 funnel.
converting_sessions AS (
  SELECT *
  FROM pivoted
  WHERE
    ts_session  IS NOT NULL
    AND ts_view     >= ts_session
    AND ts_cart     >= ts_view
    AND ts_checkout >= ts_cart
    AND ts_purchase >= ts_checkout
),
-- all_purchases: every session with a purchase, sequential or not. The funnel's
-- revenue is a subset of this; the gap is reported, not hidden.
all_purchases AS (
  SELECT
    COUNT(*)                AS all_purchasing_sessions,
    ROUND(SUM(revenue), 2)  AS all_revenue,
    COUNTIF(ts_cart IS NULL) AS purchases_without_cart
  FROM pivoted
  WHERE ts_purchase IS NOT NULL
)
SELECT c.*, all_purchases.*
FROM (
  SELECT
    COUNT(*)                                AS purchasing_sessions,
    -- Some purchase events carry no purchase_revenue: AOV averages the orders
    -- that do, and both counts are kept so every figure reconciles.
    COUNTIF(revenue > 0)                    AS orders_with_revenue,
    ROUND(SUM(revenue), 2)                  AS total_revenue,
    ROUND(AVG(IF(revenue > 0, revenue, NULL)), 2) AS avg_order_value,
    ROUND(APPROX_QUANTILES(IF(revenue > 0, revenue, NULL), 100)[OFFSET(50)], 2) AS median_order_value,
    ROUND(MIN(IF(revenue > 0, revenue, NULL)), 2) AS min_order_value,
    ROUND(MAX(revenue), 2)                  AS max_order_value
  FROM converting_sessions
) c
CROSS JOIN all_purchases
"""

revenue_df = client.query(revenue_sql).to_dataframe()
print("Revenue summary for purchasing sessions:")
display(revenue_df)

# Normalize revenue against all sessions that entered the funnel (step 1).
total_sessions = int(funnel_raw["step1_sessions"].iloc[0])
total_revenue  = float(revenue_df["total_revenue"].iloc[0])
purchases      = int(revenue_df["purchasing_sessions"].iloc[0])
orders_with_rev = int(revenue_df["orders_with_revenue"].iloc[0])
rev_per_purchase = total_revenue / purchases
print(f"\nRevenue per session entering funnel: ${total_revenue / total_sessions:.4f}")
print(f"Overall session-to-purchase rate:    {purchases / total_sessions * 100:.2f}%")
# Reconciliation: AOV × orders-with-revenue = total; revenue per purchase × purchases = total.
print(f"Funnel purchases: {purchases:,}  |  with revenue recorded: {orders_with_rev:,}  "
      f"|  without: {purchases - orders_with_rev:,}")
print(f"AOV ${revenue_df['avg_order_value'].iloc[0]:,.2f} (over {orders_with_rev:,})  |  "
      f"revenue per funnel purchase ${rev_per_purchase:,.2f} (over {purchases:,})")
print(f"All purchasing sessions (sequential or not): {int(revenue_df['all_purchasing_sessions'].iloc[0]):,}  "
      f"(${float(revenue_df['all_revenue'].iloc[0]):,.0f}); "
      f"{int(revenue_df['purchases_without_cart'].iloc[0]):,} of them have no add_to_cart in the session")


Revenue summary for purchasing sessions:


,purchasing_sessions,orders_with_revenue,total_revenue,avg_order_value,median_order_value,min_order_value,max_order_value,all_purchasing_sessions,all_revenue,purchases_without_cart
0,2721,2461,193678.0,78.7,55.0,3.0,1530.0,4848,362165.0,2000



Revenue per session entering funnel: $0.5458
Overall session-to-purchase rate:    0.77%
Funnel purchases: 2,721  |  with revenue recorded: 2,461  |  without: 260
AOV $78.70 (over 2,461)  |  revenue per funnel purchase $71.18 (over 2,721)
All purchasing sessions (sequential or not): 4,848  ($362,165); 2,000 of them have no add_to_cart in the session


## 5. Device category breakdown
Same session-scoped funnel, split by `device.category`.
Highlights whether mobile vs desktop differences are meaningful or within noise.

In [7]:
# ── Section 5: Device category breakdown (BigQuery) ─────────────────────────
# Same session-scoped sequential funnel as Section 3, grouped by device.category.
device_sql = f"""
WITH base AS (
  SELECT
    user_pseudo_id,
    device.category AS device_category,
    (SELECT ep.value.int_value FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'ga_session_id' LIMIT 1) AS session_id,
    event_name,
    event_timestamp
  FROM `{DATASET}.events_*`
  WHERE
    _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name IN ('session_start','view_item','add_to_cart','begin_checkout','purchase')
),
session_events AS (
  SELECT
    user_pseudo_id, session_id,
    ANY_VALUE(device_category) AS device_category,
    event_name,
    MIN(event_timestamp) AS first_ts
  FROM base
  WHERE session_id IS NOT NULL
  GROUP BY user_pseudo_id, session_id, event_name
),
pivoted AS (
  SELECT
    user_pseudo_id, session_id,
    ANY_VALUE(device_category)                                                   AS device_category,
    MAX(CASE WHEN event_name = 'session_start'  THEN first_ts END)               AS ts_session,
    MAX(CASE WHEN event_name = 'view_item'      THEN first_ts END)               AS ts_view,
    MAX(CASE WHEN event_name = 'add_to_cart'    THEN first_ts END)               AS ts_cart,
    MAX(CASE WHEN event_name = 'begin_checkout' THEN first_ts END)               AS ts_checkout,
    MAX(CASE WHEN event_name = 'purchase'       THEN first_ts END)               AS ts_purchase
  FROM session_events
  GROUP BY user_pseudo_id, session_id
)
SELECT
  device_category,
  COUNT(*)                                                                        AS step1_sessions,
  COUNTIF(ts_view     IS NOT NULL AND ts_view     >= ts_session)                 AS step2_view_item,
  COUNTIF(ts_cart     IS NOT NULL AND ts_cart     >= ts_view
                                 AND ts_view      >= ts_session)                 AS step3_add_to_cart,
  COUNTIF(ts_checkout IS NOT NULL AND ts_checkout >= ts_cart
                                 AND ts_cart      >= ts_view
                                 AND ts_view      >= ts_session)                 AS step4_begin_checkout,
  COUNTIF(ts_purchase IS NOT NULL AND ts_purchase >= ts_checkout
                                 AND ts_checkout  >= ts_cart
                                 AND ts_cart      >= ts_view
                                 AND ts_view      >= ts_session)                 AS step5_purchase
FROM pivoted
WHERE ts_session IS NOT NULL
GROUP BY device_category
ORDER BY step1_sessions DESC
"""

device_raw = client.query(device_sql).to_dataframe()
display(device_raw)


,device_category,step1_sessions,step2_view_item,step3_add_to_cart,step4_begin_checkout,step5_purchase
0,desktop,205899,43810,8606,2985,1537
1,mobile,141079,29793,5948,2108,1123
2,tablet,7879,1658,316,112,61


In [8]:
# ── Section 5b: Device funnel chart ───────────────────────────────────────
# Map wide SQL column names to display labels for melt/pivot operations.
step_map = {
    "step1_sessions":       "1. Session Start",
    "step2_view_item":      "2. View Item",
    "step3_add_to_cart":    "3. Add to Cart",
    "step4_begin_checkout": "4. Begin Checkout",
    "step5_purchase":       "5. Purchase",
}
# Long format: one row per (device, funnel step) for grouped bar chart.
device_long = device_raw.melt(
    id_vars="device_category",
    value_vars=list(step_map.keys()),
    var_name="step_col", value_name="sessions",
)
device_long["step"] = device_long["step_col"].map(step_map)
# Denominator for % conversion: sessions at step 1 per device.
base = (
    device_long[device_long["step_col"] == "step1_sessions"]
    [["device_category","sessions"]].rename(columns={"sessions":"base"})
)
device_long = device_long.merge(base, on="device_category")
device_long["pct"] = (device_long["sessions"] / device_long["base"] * 100).round(2)

# Grouped bar: % of each device's sessions still in funnel at each step.
fig_dev = px.bar(
    device_long, x="step", y="pct", color="device_category",
    barmode="group",
    title="Session Funnel Conversion Rate by Device",
    labels={"pct": "% of Sessions", "step": "", "device_category": "Device"},
    color_discrete_map={"desktop": "#4C78A8", "mobile": "#F58518", "tablet": "#54A24B"},
    text="pct",
)
fig_dev.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig_dev.update_layout(height=450, xaxis_tickangle=-20, uniformtext_minsize=8)
fig_dev.show()

# Absolute purchase counts and session-to-purchase % by device (LLM context uses this).
purchase_rates = device_long[device_long["step_col"] == "step5_purchase"][["device_category","sessions","base","pct"]]
print("\nSession-to-purchase rate by device:")
display(purchase_rates.reset_index(drop=True))



Session-to-purchase rate by device:


,device_category,sessions,base,pct
0,desktop,1537,205899,0.75
1,mobile,1123,141079,0.8
2,tablet,61,7879,0.77


## 5c. Acquisition channel breakdown — is discovery a traffic problem?

Most sessions never see a product (Section 3). Two very different explanations fit
that, and they lead to different work:

| If the miss rate is… | Then the problem is… | And the fix is… |
|---|---|---|
| **Uniform across channels** | On the site — navigation / discovery | Homepage and category navigation |
| **Concentrated in some channels** | Who is arriving — traffic quality | Ad targeting, landing pages, budget mix |

This section splits the same session-scoped funnel by channel. A **10pp spread** in
session → view-item rate between channels is used as the decision threshold — an
analyst-set assumption (roughly half the site's product-view rate), not an industry standard.
`(none)` is GA4's label for direct traffic; `(not set)` means the session carries no
medium at all (Section 5e shows what those sessions are).

**Grain trap — read before reusing this pattern.** `traffic_source.medium` in the
GA4 export is the **user's first-touch** medium: every session a user ever has
inherits it. Grouping a *session* funnel by it attributes a returning direct visit
to the paid campaign that first acquired the user. The session's own medium lives
in `event_params` (`key = 'medium'`), so it is taken from the earliest event in the
session that carries one. Both are returned so they can be compared.

In [9]:
# ── Section 5c: Acquisition channel breakdown (BigQuery) ─────────────────────
# Same sequential funnel as Section 3, plus revenue, grouped by BOTH the
# session-level medium and the user's first-touch medium.
channel_sql = f"""
-- base: funnel events, session id, first-touch medium (user-level), revenue.
WITH base AS (
  SELECT
    user_pseudo_id,
    (SELECT ep.value.int_value FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'ga_session_id' LIMIT 1)          AS session_id,
    traffic_source.medium                             AS first_touch_medium,
    event_name,
    event_timestamp,
    ecommerce.purchase_revenue                        AS purchase_revenue
  FROM `{DATASET}.events_*`
  WHERE
    _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name IN ('session_start','view_item','add_to_cart','begin_checkout','purchase')
),

-- session_medium: medium on the earliest event in the session that carries one.
-- Scans every event type: medium is not guaranteed on the funnel events.
session_medium AS (
  SELECT
    user_pseudo_id, session_id,
    ARRAY_AGG(medium ORDER BY event_timestamp LIMIT 1)[OFFSET(0)] AS session_medium
  FROM (
    SELECT
      user_pseudo_id, event_timestamp,
      (SELECT ep.value.int_value    FROM UNNEST(event_params) AS ep
       WHERE ep.key = 'ga_session_id' LIMIT 1) AS session_id,
      (SELECT ep.value.string_value FROM UNNEST(event_params) AS ep
       WHERE ep.key = 'medium' LIMIT 1)        AS medium
    FROM `{DATASET}.events_*`
    WHERE _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
  )
  WHERE session_id IS NOT NULL AND medium IS NOT NULL
  GROUP BY user_pseudo_id, session_id
),

session_events AS (
  SELECT
    user_pseudo_id, session_id,
    ANY_VALUE(first_touch_medium) AS first_touch_medium,
    event_name,
    MIN(event_timestamp)          AS first_ts,
    SUM(purchase_revenue)         AS rev
  FROM base
  WHERE session_id IS NOT NULL
  GROUP BY user_pseudo_id, session_id, event_name
),

pivoted AS (
  SELECT
    user_pseudo_id, session_id,
    ANY_VALUE(first_touch_medium)                                    AS first_touch_medium,
    MAX(CASE WHEN event_name = 'session_start'  THEN first_ts END)   AS ts_session,
    MAX(CASE WHEN event_name = 'view_item'      THEN first_ts END)   AS ts_view,
    MAX(CASE WHEN event_name = 'add_to_cart'    THEN first_ts END)   AS ts_cart,
    MAX(CASE WHEN event_name = 'begin_checkout' THEN first_ts END)   AS ts_checkout,
    MAX(CASE WHEN event_name = 'purchase'       THEN first_ts END)   AS ts_purchase,
    MAX(CASE WHEN event_name = 'purchase'       THEN rev END)        AS revenue
  FROM session_events
  GROUP BY user_pseudo_id, session_id
),

-- converted: flag the full sequential path once, reuse for count and revenue.
flagged AS (
  SELECT
    p.*,
    COALESCE(sm.session_medium, '(not set)') AS session_medium_label,
    (ts_purchase IS NOT NULL AND ts_purchase >= ts_checkout AND ts_checkout >= ts_cart
       AND ts_cart >= ts_view AND ts_view >= ts_session)    AS converted
  FROM pivoted AS p
  LEFT JOIN session_medium AS sm USING (user_pseudo_id, session_id)
  WHERE ts_session IS NOT NULL
)

SELECT
  session_medium_label                                     AS session_medium,
  COALESCE(first_touch_medium, '(not set)')                AS first_touch_medium,
  COUNT(*)                                                 AS sessions,
  COUNTIF(ts_view IS NOT NULL AND ts_view >= ts_session)   AS view_item,
  COUNTIF(ts_cart IS NOT NULL AND ts_cart >= ts_view
                              AND ts_view >= ts_session)   AS add_to_cart,
  COUNTIF(converted)                                       AS purchase,
  ROUND(SUM(IF(converted, IFNULL(revenue, 0), 0)), 2)      AS revenue
FROM flagged
GROUP BY session_medium, first_touch_medium
"""

channel_raw = client.query(channel_sql).to_dataframe()

# Coverage: if most sessions have no session-level medium, fall back to first touch.
coverage = 1 - channel_raw.loc[channel_raw["session_medium"] == "(not set)", "sessions"].sum() / channel_raw["sessions"].sum()
channel_dim = "session_medium" if coverage >= 0.5 else "first_touch_medium"
print(f"Session-level medium coverage: {coverage*100:.1f}%  →  using {channel_dim}")

# Reconciliation: every session lands in exactly one channel.
assert channel_raw["sessions"].sum() == funnel_raw["step1_sessions"].iloc[0], "channel join dropped/duplicated sessions"

# How often do the two definitions disagree? This is the grain trap, quantified.
agree = channel_raw.loc[channel_raw["session_medium"] == channel_raw["first_touch_medium"], "sessions"].sum()
print(f"Sessions where session medium == first-touch medium: {agree / channel_raw['sessions'].sum() * 100:.1f}%")
display(channel_raw.pivot_table(index="first_touch_medium", columns="session_medium",
                                values="sessions", aggfunc="sum", fill_value=0))

Session-level medium coverage: 73.8%  →  using session_medium
Sessions where session medium == first-touch medium: 52.9%


session_medium,(data deleted),(none),(not set),<Other>,affiliate,cpc,email,organic,referral
first_touch_medium,,,,,,,,,
(data deleted),6315,0,4269,0,150,0,24,3177,7527
(none),0,37670,21863,0,309,1,49,10338,12132
<Other>,0,0,14261,25302,150,1,23,6479,5560
cpc,0,0,4246,0,43,7659,7,1909,1671
organic,0,0,32735,0,406,2,46,72732,15645
referral,0,0,15518,0,251,1,43,8153,38190


In [10]:
# ── Section 5d: Channel funnel metrics + chart ──────────────────────────────
metrics = ["sessions", "view_item", "add_to_cart", "purchase", "revenue"]
channel_df = (channel_raw.groupby(channel_dim, as_index=False)[metrics].sum()
              .rename(columns={channel_dim: "channel"}))

# Group channels under 1% of sessions — tiny denominators produce noisy rates.
total = channel_df["sessions"].sum()
small = channel_df["sessions"] / total < 0.01
if small.sum() > 1:
    other = channel_df[small][metrics].sum().to_frame().T
    other.insert(0, "channel", f"other ({int(small.sum())} small channels)")
    channel_df = pd.concat([channel_df[~small], other], ignore_index=True)

channel_df[metrics]            = channel_df[metrics].astype(float)
channel_df["share_pct"]        = channel_df["sessions"] / total * 100
channel_df["view_pct"]         = channel_df["view_item"] / channel_df["sessions"] * 100
channel_df["cart_from_view"]   = channel_df["add_to_cart"] / channel_df["view_item"].where(channel_df["view_item"] > 0) * 100
channel_df["purchase_pct"]     = channel_df["purchase"] / channel_df["sessions"] * 100
channel_df["rev_per_session"]  = channel_df["revenue"] / channel_df["sessions"]
channel_df["low_n"]            = channel_df["purchase"] < 30   # purchase rate too noisy to rank
channel_df = channel_df.sort_values("sessions", ascending=False).reset_index(drop=True)
display(channel_df.style.format({
    "sessions": "{:,.0f}", "share_pct": "{:.1f}%", "view_pct": "{:.1f}%",
    "cart_from_view": "{:.1f}%", "purchase_pct": "{:.2f}%", "rev_per_session": "${:,.2f}",
    "view_item": "{:,.0f}", "add_to_cart": "{:,.0f}", "purchase": "{:,.0f}", "revenue": "${:,.0f}",
}))

site_view_rate = funnel_df.loc[funnel_df["step"] == "2. View Products", "overall_cvr"].iloc[0]
plot_df = channel_df.sort_values("view_pct")
fig_ch = go.Figure(go.Bar(
    y=plot_df["channel"], x=plot_df["view_pct"].round(1), orientation="h",
    text=[f"{v:.1f}%" for v in plot_df["view_pct"]], textposition="outside",
    marker_color=["#dc2626" if v < site_view_rate - 5 else "#4C78A8" for v in plot_df["view_pct"]],
))
fig_ch.add_vline(x=site_view_rate, line_dash="dash", line_color="#6b7280",
                 annotation_text=f"site avg {site_view_rate:.1f}%", annotation_position="top")
fig_ch.update_layout(title=f"Session → View Item Rate by Channel  ({channel_dim})",
                     height=max(300, 46 * len(plot_df) + 100), showlegend=False,
                     xaxis_title="% of sessions that saw a product")
fig_ch.show()

# Decision rule: spread across actionable channels (excludes the 'other' bucket and '(not set)').
ranked = channel_df[~channel_df["channel"].str.startswith(("other (", "(not set)"))]
spread = ranked["view_pct"].max() - ranked["view_pct"].min()
best, worst = ranked.loc[ranked["view_pct"].idxmax()], ranked.loc[ranked["view_pct"].idxmin()]
print(f"View-item spread: {spread:.1f}pp  ({worst['channel']} {worst['view_pct']:.1f}% → {best['channel']} {best['view_pct']:.1f}%)")
# Within 3pp of the threshold the verdict is borderline, not settled.
if spread >= 10:
    verdict = "traffic mix is a driver — check landing pages / targeting for the weakest channel"
elif spread >= 7:
    verdict = (f"borderline — leaning 'the leak is on the site', but close to the threshold; "
               f"review {worst['channel']} landing pages before a navigation redesign")
else:
    verdict = "uniform across channels — the leak is on the site (navigation / discovery)"
print("Verdict:", verdict)
print("Note: the low_n flag applies to purchase_pct / revenue only — view_pct rests on session counts.")

# (not set) is excluded from the spread but is a large block — disclose it.
ns = channel_df[channel_df["channel"] == "(not set)"]
if len(ns):
    print(f"(not set): {ns['share_pct'].iloc[0]:.1f}% of sessions, view rate {ns['view_pct'].iloc[0]:.1f}% "
          f"(site average {site_view_rate:.1f}%) — a tagging gap in short sessions, not a channel; see 5e")

,channel,sessions,view_item,add_to_cart,purchase,revenue,share_pct,view_pct,cart_from_view,purchase_pct,rev_per_session,low_n
0,organic,"102,788","27,055","5,191",930,"$59,421",29.0%,26.3%,19.2%,0.90%,$0.58,False
1,(not set),"92,892","6,205",564,1,$12,26.2%,6.7%,9.1%,0.00%,$0.00,True
2,referral,"80,725","24,443","5,838","1,385","$103,656",22.7%,30.3%,23.9%,1.72%,$1.28,False
3,(none),"37,670","8,543","1,576",182,"$14,403",10.6%,22.7%,18.4%,0.48%,$0.38,False
4,,"25,302","5,608",954,66,"$5,381",7.1%,22.2%,17.0%,0.26%,$0.21,False
5,cpc,"7,664","1,653",268,28,"$2,009",2.2%,21.6%,16.2%,0.37%,$0.26,True
6,(data deleted),"6,315","1,455",401,109,"$7,042",1.8%,23.0%,27.6%,1.73%,$1.12,False
7,other (2 small channels),"1,501",299,78,20,"$1,754",0.4%,19.9%,26.1%,1.33%,$1.17,True


View-item spread: 8.7pp  (cpc 21.6% → referral 30.3%)
Verdict: borderline — leaning 'the leak is on the site', but close to the threshold; review cpc landing pages before a navigation redesign
Note: the low_n flag applies to purchase_pct / revenue only — view_pct rests on session counts.
(not set): 26.2% of sessions, view rate 6.7% (site average 21.2%) — a tagging gap in short sessions, not a channel; see 5e


## 5e. Data quality — what `(not set)` is, and what `view_item` records

**`(not set)` sessions.** About a quarter of sessions carry no session-level `medium`.
Before treating them as bots or excluding them, test what they are: the `medium` param is
attached to only *some* events, so a short session is more likely to carry none by chance.
If that is the mechanism, then at the **same session length** their product-view rate should
match attributed sessions — and excluding them would remove short sessions selectively and
flatter every funnel rate. All sessions therefore stay the primary view; the
excluding-`(not set)` funnel is a sensitivity check only.

**`view_item` / `add_to_cart` semantics.** If these events carry a whole product list in
`items`, they are listing impressions / quick-adds, not product-page views — and on
`add_to_cart` only the item with `quantity > 0` was actually carted (used in Section 7c).

In [11]:
# ── Section 5e: Data quality — unattributed sessions and list-level events ──
# Same SQL as generate_dashboard.py, so notebook and dashboard agree. The same
# per-session scan carries ga_session_number for the new vs returning split (8c).
dq_sql = f"""
WITH ev AS (
  SELECT user_pseudo_id, event_name, event_timestamp, device.category AS dev,
    ecommerce.purchase_revenue AS purchase_revenue,
    (SELECT ep.value.int_value FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'ga_session_id' LIMIT 1) AS session_id,
    (SELECT ep.value.string_value FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'medium' LIMIT 1) AS medium,
    (SELECT ep.value.int_value FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'engagement_time_msec' LIMIT 1) AS eng_msec,
    (SELECT ep.value.int_value FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'ga_session_number' LIMIT 1) AS session_number
  FROM `{DATASET}.events_*`
  WHERE _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
),
s AS (
  SELECT user_pseudo_id, session_id, ANY_VALUE(dev) AS dev,
    COUNT(*) AS n_events, COUNTIF(medium IS NOT NULL) AS n_medium,
    COUNTIF(event_name = 'page_view') AS n_page_view,
    SUM(IFNULL(eng_msec, 0)) / 1000 AS engaged_s,
    -- GA4 counts sessions per user from the first visit ever, not from the
    -- start of this window, so 1 = genuinely new (for that cookie).
    MAX(session_number) AS session_number,
    MIN(IF(event_name='session_start',  event_timestamp, NULL)) AS ts_s,
    MIN(IF(event_name='view_item',      event_timestamp, NULL)) AS ts_v,
    MIN(IF(event_name='add_to_cart',    event_timestamp, NULL)) AS ts_c,
    MIN(IF(event_name='begin_checkout', event_timestamp, NULL)) AS ts_k,
    MIN(IF(event_name='purchase',       event_timestamp, NULL)) AS ts_p,
    SUM(IF(event_name='purchase', purchase_revenue, NULL))      AS revenue
  FROM ev WHERE session_id IS NOT NULL
  GROUP BY user_pseudo_id, session_id
),
f AS (
  SELECT *,
    n_medium > 0 AS attributed,
    CASE WHEN session_number = 1 THEN 'new' WHEN session_number > 1 THEN 'returning'
         ELSE 'unknown' END AS visitor,
    CASE WHEN n_events <= 3 THEN '1–3' WHEN n_events <= 5 THEN '4–5'
         WHEN n_events <= 8 THEN '6–8' WHEN n_events <= 15 THEN '9–15'
         ELSE '16+' END AS size_bucket,
    IFNULL(ts_v >= ts_s, FALSE) AS s2,
    IFNULL(ts_c >= ts_v AND ts_v >= ts_s, FALSE) AS s3,
    IFNULL(ts_k >= ts_c AND ts_c >= ts_v AND ts_v >= ts_s, FALSE) AS s4,
    IFNULL(ts_p >= ts_k AND ts_k >= ts_c AND ts_c >= ts_v AND ts_v >= ts_s, FALSE) AS s5
  FROM s WHERE ts_s IS NOT NULL
)
SELECT
  CASE WHEN GROUPING(size_bucket) = 0 THEN 'size'
       WHEN GROUPING(dev) = 0 THEN 'device'
       WHEN GROUPING(visitor) = 0 THEN 'visitor' ELSE 'all' END AS cut,
  size_bucket, dev, attributed, visitor,
  COUNT(*) AS sessions,
  COUNTIF(s2) AS view_item, COUNTIF(s3) AS add_to_cart,
  COUNTIF(s4) AS begin_checkout, COUNTIF(s5) AS purchase,
  ROUND(SUM(IF(s5, IFNULL(revenue, 0), 0)), 2) AS revenue,
  COUNTIF(n_page_view > 0) AS with_page_view,
  APPROX_QUANTILES(engaged_s, 2)[OFFSET(1)] AS median_engaged_s,
  APPROX_QUANTILES(n_events, 2)[OFFSET(1)] AS median_events
FROM f
GROUP BY GROUPING SETS ((attributed), (size_bucket, attributed), (dev, attributed), (visitor))
"""
dq_raw = client.query(dq_sql).to_dataframe()

dq_all = dq_raw[dq_raw["cut"] == "all"].set_index("attributed")
ua, at = dq_all.loc[False], dq_all.loc[True]
n_all  = int(dq_all["sessions"].sum())
ua_share      = ua["sessions"] / n_all * 100
ua_view_pct   = ua["view_item"] / ua["sessions"] * 100
at_view_pct   = at["view_item"] / at["sessions"] * 100
ua_lost_share = (ua["sessions"] - ua["view_item"]) / (n_all - int(dq_all["view_item"].sum())) * 100
print(f"(not set): {int(ua['sessions']):,} sessions ({ua_share:.1f}%)  |  saw a product "
      f"{ua_view_pct:.1f}% vs {at_view_pct:.1f}% attributed")
print(f"Median events {int(ua['median_events'])} vs {int(at['median_events'])}  |  median engaged "
      f"{ua['median_engaged_s']:.0f}s vs {at['median_engaged_s']:.0f}s  |  with page_view "
      f"{ua['with_page_view']/ua['sessions']*100:.0f}% vs {at['with_page_view']/at['sessions']*100:.0f}%")
print(f"Share of the never-saw-a-product loss inside (not set): {ua_lost_share:.1f}%")

# Session-length test: at matched length, do the two groups behave alike?
_sz = (dq_raw[dq_raw["cut"] == "size"]
       .pivot(index="size_bucket", columns="attributed", values=["sessions", "view_item"])
       .reindex(["1–3", "4–5", "6–8", "9–15", "16+"]))
size_tbl = pd.DataFrame({
    "sessions":            _sz[("sessions", True)] + _sz[("sessions", False)],
    "unattributed_pct":    _sz[("sessions", False)] / (_sz[("sessions", True)] + _sz[("sessions", False)]) * 100,
    "saw_product_unattr":  _sz[("view_item", False)] / _sz[("sessions", False)] * 100,
    "saw_product_attr":    _sz[("view_item", True)]  / _sz[("sessions", True)]  * 100,
})
size_tbl.index.name = "events in session"
display(size_tbl.style.format({"sessions": "{:,.0f}", "unattributed_pct": "{:.1f}%",
                                "saw_product_unattr": "{:.1f}%", "saw_product_attr": "{:.1f}%"}))

_dv = dq_raw[dq_raw["cut"] == "device"].pivot(index="dev", columns="attributed", values="sessions")
print("Unattributed share by device:",
      ", ".join(f"{d} {v:.1f}%" for d, v in (_dv[False] / _dv.sum(axis=1) * 100).items()))

# Sensitivity funnel — NOT the primary view (biased upward, see markdown above).
def _stats(r):
    return {"sessions": int(r["sessions"]),
            "saw_product_pct": r["view_item"] / r["sessions"] * 100,
            "saw_to_cart_pct": r["add_to_cart"] / r["view_item"] * 100,
            "cart_to_checkout_pct": r["begin_checkout"] / r["add_to_cart"] * 100,
            "checkout_to_purchase_pct": r["purchase"] / r["begin_checkout"] * 100,
            "session_to_purchase_pct": r["purchase"] / r["sessions"] * 100,
            "revenue_per_session": r["revenue"] / r["sessions"]}
sensitivity_df = pd.DataFrame({"all sessions (primary)": _stats(dq_all.sum(numeric_only=True)),
                               "excluding (not set) — biased up": _stats(at)})
display(sensitivity_df.round(2))

# Event semantics: list-sized items arrays → listing impressions / quick-adds.
evq_sql = f"""
SELECT event_name, COUNT(*) AS events,
  ROUND(AVG(ARRAY_LENGTH(items)), 1) AS avg_items,
  COUNTIF(ARRAY_LENGTH(items) > 1) AS multi_item_events,
  COUNTIF((SELECT COUNTIF(IFNULL(it.quantity, 0) > 0) FROM UNNEST(items) AS it) = 1) AS one_carted_item_events
FROM `{DATASET}.events_*`
WHERE _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
  AND event_name IN ('view_item', 'add_to_cart')
GROUP BY event_name
"""
evq_raw = client.query(evq_sql).to_dataframe().set_index("event_name")
vi_multi_pct = evq_raw.loc["view_item", "multi_item_events"] / evq_raw.loc["view_item", "events"] * 100
atc_one_pct  = evq_raw.loc["add_to_cart", "one_carted_item_events"] / evq_raw.loc["add_to_cart", "events"] * 100
display(evq_raw)
print(f"view_item events with several products: {vi_multi_pct:.0f}% (avg {evq_raw.loc['view_item', 'avg_items']} items)")
print(f"add_to_cart events with exactly one item carrying a quantity: {atc_one_pct:.0f}% "
      f"(avg {evq_raw.loc['add_to_cart', 'avg_items']} items attached)")

(not set): 92,892 sessions (26.2%)  |  saw a product 6.7% vs 26.4% attributed
Median events 4 vs 6  |  median engaged 2s vs 13s  |  with page_view 94% vs 92%
Share of the never-saw-a-product loss inside (not set): 31.0%


,sessions,unattributed_pct,saw_product_unattr,saw_product_attr
events in session,,,,
1–3,"64,233",38.0%,1.5%,1.3%
4–5,"131,833",37.5%,5.1%,7.3%
6–8,"61,201",25.0%,12.1%,17.9%
9–15,"44,714",7.5%,38.7%,34.0%
16+,"52,876",0.6%,53.1%,76.5%


Unattributed share by device: desktop 26.2%, mobile 26.2%, tablet 26.2%


,all sessions (primary),excluding (not set) — biased up
sessions,354857.00,261965.00
saw_product_pct,21.21,26.36
saw_to_cart_pct,19.76,20.72
cart_to_checkout_pct,35.00,36.36
checkout_to_purchase_pct,52.28,52.29
session_to_purchase_pct,0.77,1.04
revenue_per_session,0.55,0.74


,events,avg_items,multi_item_events,one_carted_item_events
event_name,,,,
add_to_cart,58543,11.4,56694,37257
view_item,386068,7.1,233150,19125


view_item events with several products: 60% (avg 7.1 items)
add_to_cart events with exactly one item carrying a quantity: 64% (avg 11.4 items attached)


## 5f. Statistical significance — device and channel (E6)

**Device.** Wilson 95% intervals per device and a chi-square test of purchase × device. With
rates this close, the expected result is *not significant* — the formal version of "device is
not the lever".

**Channel.** The same test on "saw a product" across named channels. With ~260k sessions almost
any gap is significant, so the result is reported with an effect size (Cramér's V). Whether the
discovery leak is the site or the traffic is a question of **practical size** — the 10pp
threshold — not of significance.

Intervals and tests come from `stats_utils.py`, shared with the dashboard.

In [12]:
# ── Section 5f: Statistical significance — device and channel (E6) ──────────
from stats_utils import wilson_ci, chi_square, format_p

dev_sig = device_raw[["device_category", "step1_sessions", "step5_purchase"]].copy()
dev_sig["purchase_pct"] = dev_sig["step5_purchase"] / dev_sig["step1_sessions"] * 100
dev_sig[["ci_lo", "ci_hi"]] = [wilson_ci(p, n) for p, n in zip(dev_sig["step5_purchase"], dev_sig["step1_sessions"])]
display(dev_sig.style.format({"step1_sessions": "{:,}", "step5_purchase": "{:,}",
                              "purchase_pct": "{:.2f}%", "ci_lo": "{:.2f}%", "ci_hi": "{:.2f}%"}))
device_test = chi_square(dev_sig["step5_purchase"], dev_sig["step1_sessions"])
print(f"Device × purchase: chi2 = {device_test['chi2']:.2f}, df = {device_test['dof']}, "
      f"{format_p(device_test['p'])}, Cramér's V = {device_test['cramers_v']:.4f}  →  "
      + ("significant (check the effect size)" if device_test["p"] < 0.05 else "not significant"))

channel_test = chi_square(ranked["view_item"], ranked["sessions"])
print(f"Channel × saw-a-product (named channels): chi2 = {channel_test['chi2']:,.0f}, "
      f"df = {channel_test['dof']}, {format_p(channel_test['p'])}, Cramér's V = {channel_test['cramers_v']:.3f}")
print("→ significance is expected at this volume; the site-vs-traffic verdict rests on the 10pp practical threshold.")

,device_category,step1_sessions,step5_purchase,purchase_pct,ci_lo,ci_hi
0,desktop,"205,899","1,537",0.75%,0.71%,0.78%
1,mobile,"141,079","1,123",0.80%,0.75%,0.84%
2,tablet,"7,879",61,0.77%,0.60%,0.99%


Device × purchase: chi2 = 2.70, df = 2, p = 0.259, Cramér's V = 0.0028  →  not significant
Channel × saw-a-product (named channels): chi2 = 1,257, df = 5, p < 0.001, Cramér's V = 0.069
→ significance is expected at this volume; the site-vs-traffic verdict rests on the 10pp practical threshold.


## 6. Time-to-convert
For sessions that **complete the full funnel**, the median time in **minutes** between each
consecutive step. It describes buyers only — it says nothing about sessions that abandon.

**Tracking check.** The same query counts carting sessions whose first `add_to_cart` has the
*identical* timestamp as their first `view_item`, or comes before / without one. Those are
instrumentation signals (quick-add on a listing firing both events), not shopper speed.

In [13]:
# ── Section 6: Time-to-convert (BigQuery) ─────────────────────────────────
# Same SQL as generate_dashboard.py. Gaps are in minutes (microseconds / 6e7).
ttc_sql = f"""
WITH base AS (
  SELECT user_pseudo_id,
    (SELECT ep.value.int_value FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'ga_session_id' LIMIT 1) AS session_id,
    event_name, event_timestamp
  FROM `{DATASET}.events_*`
  WHERE _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name IN ('session_start','view_item','add_to_cart','begin_checkout','purchase')
),
se AS (
  SELECT user_pseudo_id, session_id, event_name, MIN(event_timestamp) AS first_ts
  FROM base WHERE session_id IS NOT NULL GROUP BY user_pseudo_id, session_id, event_name
),
pv AS (
  SELECT user_pseudo_id, session_id,
    MAX(CASE WHEN event_name='session_start'  THEN first_ts END) AS ts_s,
    MAX(CASE WHEN event_name='view_item'      THEN first_ts END) AS ts_v,
    MAX(CASE WHEN event_name='add_to_cart'    THEN first_ts END) AS ts_c,
    MAX(CASE WHEN event_name='begin_checkout' THEN first_ts END) AS ts_k,
    MAX(CASE WHEN event_name='purchase'       THEN first_ts END) AS ts_p
  FROM se GROUP BY user_pseudo_id, session_id
),
ff AS (SELECT * FROM pv
  WHERE ts_s IS NOT NULL AND ts_v>=ts_s AND ts_c>=ts_v AND ts_k>=ts_c AND ts_p>=ts_k),
-- Tracking check over every session with an add_to_cart (not only converters):
-- an add-to-cart logged at the exact microsecond of the first view_item, or
-- before / without any view_item, is an instrumentation signal, not behaviour.
cart AS (
  SELECT COUNT(*) AS carting_sessions,
    COUNTIF(ts_c = ts_v) AS cart_same_ts_as_view,
    COUNTIF(ts_v IS NULL OR ts_c < ts_v) AS cart_before_or_without_view,
    COUNTIF(ts_c > ts_v AND ts_c - ts_v < 1000000) AS cart_within_1s_after_view
  FROM pv WHERE ts_c IS NOT NULL
)
SELECT t.*, cart.*
FROM (
  -- Minutes: the steps are minutes apart, so hours hide them behind decimals.
  SELECT
    ROUND(APPROX_QUANTILES((ts_v-ts_s)/60000000.0,100)[OFFSET(50)],1) AS p50_s_to_v,
    ROUND(APPROX_QUANTILES((ts_c-ts_v)/60000000.0,100)[OFFSET(50)],1) AS p50_v_to_c,
    ROUND(APPROX_QUANTILES((ts_k-ts_c)/60000000.0,100)[OFFSET(50)],1) AS p50_c_to_k,
    ROUND(APPROX_QUANTILES((ts_p-ts_k)/60000000.0,100)[OFFSET(50)],1) AS p50_k_to_p,
    ROUND(APPROX_QUANTILES((ts_p-ts_s)/60000000.0,100)[OFFSET(50)],1) AS p50_total,
    COUNT(*) AS converting_sessions,
    COUNTIF(ts_c = ts_v) AS converting_same_ts
  FROM ff
) t CROSS JOIN cart
"""

ttc_df = client.query(ttc_sql).to_dataframe()
print("Median minutes between funnel steps — converting sessions only:")
display(ttc_df[["p50_s_to_v", "p50_v_to_c", "p50_c_to_k", "p50_k_to_p", "p50_total", "converting_sessions"]].T
        .rename(columns={0: "value"}))

cart_same_ts_pct = ttc_df["cart_same_ts_as_view"].iloc[0] / ttc_df["carting_sessions"].iloc[0] * 100
print(f"Carting sessions: {int(ttc_df['carting_sessions'].iloc[0]):,}  |  first add_to_cart at the exact "
      f"timestamp of the first view_item: {int(ttc_df['cart_same_ts_as_view'].iloc[0]):,} ({cart_same_ts_pct:.0f}%)  |  "
      f"strictly within 1s: {int(ttc_df['cart_within_1s_after_view'].iloc[0]):,}  |  "
      f"before / without view_item: {int(ttc_df['cart_before_or_without_view'].iloc[0]):,}")
print("→ a tracking signal (listing quick-add), not evidence of 'decisive shoppers'.")

Median minutes between funnel steps — converting sessions only:


,value
p50_s_to_v,1.0
p50_v_to_c,0.3
p50_c_to_k,6.7
p50_k_to_p,3.3
p50_total,15.7
converting_sessions,2721


Carting sessions: 15,188  |  first add_to_cart at the exact timestamp of the first view_item: 2,263 (15%)  |  strictly within 1s: 0  |  before / without view_item: 42
→ a tracking signal (listing quick-add), not evidence of 'decisive shoppers'.


In [14]:
# ── Section 6b: Time-to-convert bar chart ─────────────────────────────────
ttc_steps = pd.DataFrame({
    "transition": [
        "Session → View Products",
        "View Products → Add to Cart",
        "Add to Cart → Checkout",
        "Checkout → Purchase",
    ],
    "median_min": [float(ttc_df[c].iloc[0]) for c in ("p50_s_to_v", "p50_v_to_c", "p50_c_to_k", "p50_k_to_p")],
})

fig_ttc = px.bar(
    ttc_steps, x="transition", y="median_min",
    title="Median Minutes Between Funnel Steps (Converting Sessions)",
    labels={"median_min": "Median minutes", "transition": ""},
    text="median_min", color_discrete_sequence=["#4C78A8"],
)
fig_ttc.update_traces(texttemplate="%{text:.1f} min", textposition="outside")
fig_ttc.update_layout(height=400, showlegend=False)
fig_ttc.show()

## 7. Shoppers funnel — alternative entry at `view_item`
Starting at `view_item` keeps only sessions that **saw products** (on a listing or elsewhere —
see 5e). It gives the merchandising team the conversion rate of the steps it owns.

**Read with care.** The add-to-cart rate here and the add-to-cart rate over *all* sessions have
the same numerator over two different denominators; their ratio only restates the discovery loss
from Section 3. It cannot show that product presentation is fine — that is tested per category
in Section 7c.

In [15]:
# ── Section 7: Shoppers funnel — entry at view_item (BigQuery) ────────────
# Alternative funnel: only sessions where the user viewed a product.
# Removes sessions that never saw a product — isolates shopping intent.
shoppers_sql = f"""
WITH base AS (
  SELECT
    user_pseudo_id,
    (SELECT ep.value.int_value FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'ga_session_id' LIMIT 1) AS session_id,
    event_name,
    event_timestamp
  FROM `{DATASET}.events_*`
  WHERE
    _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name IN ('view_item','add_to_cart','begin_checkout','purchase')
),
session_events AS (
  SELECT user_pseudo_id, session_id, event_name, MIN(event_timestamp) AS first_ts
  FROM base
  WHERE session_id IS NOT NULL
  GROUP BY user_pseudo_id, session_id, event_name
),
pivoted AS (
  SELECT
    user_pseudo_id, session_id,
    MAX(CASE WHEN event_name = 'view_item'      THEN first_ts END) AS ts_view,
    MAX(CASE WHEN event_name = 'add_to_cart'    THEN first_ts END) AS ts_cart,
    MAX(CASE WHEN event_name = 'begin_checkout' THEN first_ts END) AS ts_checkout,
    MAX(CASE WHEN event_name = 'purchase'       THEN first_ts END) AS ts_purchase
  FROM session_events
  GROUP BY user_pseudo_id, session_id
)
SELECT
  COUNT(*)                                                                        AS step1_view_item,
  COUNTIF(ts_cart     IS NOT NULL AND ts_cart     >= ts_view)                    AS step2_add_to_cart,
  COUNTIF(ts_checkout IS NOT NULL AND ts_checkout >= ts_cart
                                 AND ts_cart      >= ts_view)                    AS step3_begin_checkout,
  COUNTIF(ts_purchase IS NOT NULL AND ts_purchase >= ts_checkout
                                 AND ts_checkout  >= ts_cart
                                 AND ts_cart      >= ts_view)                    AS step4_purchase
FROM pivoted
WHERE ts_view IS NOT NULL
"""

shoppers_raw = client.query(shoppers_sql).to_dataframe()
print("Shoppers funnel (entry at view_item):")
display(shoppers_raw)


Shoppers funnel (entry at view_item):


,step1_view_item,step2_add_to_cart,step3_begin_checkout,step4_purchase
0,77020,15146,5293,2751


In [16]:
# ── Section 7b: Shoppers funnel metrics + comparison ──────────────────────
shopper_labels = ["1. View Products", "2. Add to Cart", "3. Begin Checkout", "4. Purchase"]
shopper_cols   = ["step1_view_item","step2_add_to_cart","step3_begin_checkout","step4_purchase"]

shoppers_df = pd.DataFrame({
    "step":     shopper_labels,
    "sessions": [int(shoppers_raw[c].iloc[0]) for c in shopper_cols],
})
shoppers_df["step_cvr"]    = (shoppers_df["sessions"] / shoppers_df["sessions"].shift(1) * 100).round(1)
shoppers_df["overall_cvr"] = (shoppers_df["sessions"] / shoppers_df["sessions"].iloc[0] * 100).round(1)
shoppers_df["drop_off"]    = (100 - shoppers_df["step_cvr"]).round(1)

print("Shoppers funnel conversion rates:")
display(shoppers_df)

fig_shop = px.funnel(
    shoppers_df, x="sessions", y="step",
    title="Shoppers Funnel (Entry: View Products)",
    color_discrete_sequence=["#54A24B"],
)
for i, row in shoppers_df.iterrows():
    if i == 0:
        continue
    fig_shop.add_annotation(
        x=row["sessions"], y=row["step"],
        text=f"  ↓ {row['drop_off']}% drop-off  ({row['step_cvr']}% pass)",
        showarrow=False, xanchor="left",
        font=dict(size=11, color="#d62728"),
    )
fig_shop.update_layout(height=400, margin=dict(l=160, r=260, t=60, b=40))
fig_shop.show()

# Compare add-to-cart from session start vs from product viewers (diagnostic split).
full_cart_rate    = funnel_df.loc[funnel_df["step"]=="3. Add to Cart", "overall_cvr"].values[0]
shopper_cart_rate = shoppers_df.loc[shoppers_df["step"]=="2. Add to Cart", "step_cvr"].values[0]
print(f"\nAdd-to-cart rate from session start:      {full_cart_rate:.1f}%")
print(f"Add-to-cart rate from sessions that saw:  {shopper_cart_rate:.1f}%")
print("Same carts over two denominators — the gap restates the discovery loss; it does not")
print(f"clear product presentation. {100 - shopper_cart_rate:.0f}% of sessions that saw products carted "
      "nothing: see 7c for where.")


Shoppers funnel conversion rates:


,step,sessions,step_cvr,overall_cvr,drop_off
0,1. View Products,77020,NaN,100.0,NaN
1,2. Add to Cart,15146,19.7,19.7,80.3
2,3. Begin Checkout,5293,34.9,6.9,65.1
3,4. Purchase,2751,52.0,3.6,48.0



Add-to-cart rate from session start:      4.2%
Add-to-cart rate from sessions that saw:  19.7%
Same carts over two denominators — the gap restates the discovery loss; it does not
clear product presentation. 80% of sessions that saw products carted nothing: see 7c for where.


## 7c. Product category — where seen → carted breaks down

The shoppers funnel shows *how much* product interest fails to become intent.
This section shows *where*: the seen → carted rate for each `item_category`,
so the merchandising audit starts from a ranked list rather than a guess.

**Grain:** one row = one (session, item_category) pair, built from `UNNEST(items)`.
A session *saw* a category when a product from it appeared in a viewed product list
(`view_item` is list-level here — Section 5e). It counts as *carting* the category only if the
item **actually carted** — the one with `quantity > 0`, not the rest of the attached list — is
from that category, at or after the first sighting in the same session.

**Why a separate query, not another funnel split?** The `items` array exists only
on `view_item`, `add_to_cart` and `purchase` — `session_start` and `begin_checkout`
carry no category — so category cannot be a dimension of the session funnel CTE.

**Label normalisation.** `item_category` mixes flat labels (`Bags`) with paths
(`Lifestyle/Bags/`, `Home/Apparel/`). Ranked raw, the same category appears several
times with its volume split. Labels are reduced to their last path segment in SQL —
before the per-session grouping, so a session that viewed both labels counts once —
and the merges are listed for review.

**Reading the numbers**
- Rates are stricter than the shoppers ATC rate (adding a *different* category's item doesn't count).
- Rows are not additive: one session can view several categories.
- A category is flagged **below** only when its whole Wilson 95% interval lies under the site
  average (**above** when wholly over it; otherwise *≈ average*). With ~23 intervals at 95%,
  about one flag may be chance — the list is a shortlist.
- Ranking uses **carts at stake** = (site average − category rate) × sessions that saw it,
  counted for flagged-below categories only,
  where the site average is the pooled rate across ranked categories (internal, not external),
  so a high-traffic category with a moderate gap outranks a niche one with a dramatic rate.

In [17]:
# ── Section 7c: Product category view → cart (BigQuery) ──────────────────────
# One row per (session, item_category). MAX(cart) >= MIN(view) means "at least
# one add-to-cart of this category at or after the first view of it".
category_sql = f"""
-- item_events: one row per item on each view_item / add_to_cart event.
WITH item_events AS (
  SELECT
    user_pseudo_id,
    (SELECT ep.value.int_value FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'ga_session_id' LIMIT 1)                     AS session_id,
    event_name,
    event_timestamp,
    COALESCE(NULLIF(TRIM(it.item_category), ''), '(not set)')    AS raw_category,
    -- add_to_cart carries the whole list (~11 items); only quantity > 0 was carted.
    IFNULL(it.quantity, 0)                                        AS quantity,
    it.price                                                      AS price
  FROM `{DATASET}.events_*`, UNNEST(items) AS it
  WHERE
    _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name IN ('view_item', 'add_to_cart')
),

-- normalised: item_category mixes flat labels ("Bags") with paths ("Lifestyle/Bags/").
-- Keep the last path segment. Done BEFORE the per-session grouping so a session that
-- viewed both "Bags" and "Lifestyle/Bags/" counts once, not twice.
-- ' / ' (slash with spaces) is part of a name, e.g. "Men's / Unisex", not a path separator.
normalised AS (
  SELECT
    *,
    IF(raw_category = '(not set)', raw_category,
       COALESCE(NULLIF(TRIM(REPLACE(ARRAY_REVERSE(SPLIT(TRIM(REPLACE(raw_category, ' / ', ' | '), '/'), '/'))[SAFE_OFFSET(0)], ' | ', ' / ')), ''),
                '(not set)'))                                     AS item_category
  FROM item_events
),

-- labels: audit trail of which raw labels were merged into each category.
labels AS (
  SELECT
    item_category,
    COUNT(DISTINCT raw_category)                                    AS n_source_labels,
    STRING_AGG(DISTINCT raw_category, ' · ' ORDER BY raw_category)  AS source_labels
  FROM normalised
  GROUP BY item_category
),

-- session_category: first view and last add-to-cart per (session, category).
session_category AS (
  SELECT
    user_pseudo_id, session_id, item_category,
    MIN(IF(event_name = 'view_item',   event_timestamp, NULL))   AS ts_first_view,
    MAX(IF(event_name = 'add_to_cart' AND quantity > 0, event_timestamp, NULL)) AS ts_last_cart
  FROM normalised
  WHERE session_id IS NOT NULL
  GROUP BY user_pseudo_id, session_id, item_category
),

agg AS (
  SELECT
    item_category,
    COUNTIF(ts_first_view IS NOT NULL)                                  AS viewing_sessions,
    COUNTIF(ts_first_view IS NOT NULL AND ts_last_cart >= ts_first_view) AS carting_sessions
  FROM session_category
  GROUP BY item_category
),

-- price: median listed price of the category's products as seen in product lists.
-- Price is a confounder for seen → carted (cheap items are an easier yes).
price AS (
  -- Exact median (PERCENTILE_CONT): APPROX_QUANTILES varies slightly between runs.
  SELECT DISTINCT item_category,
    PERCENTILE_CONT(price, 0.5) OVER (PARTITION BY item_category) AS median_price
  FROM normalised
  WHERE event_name = 'view_item' AND price IS NOT NULL
)

SELECT agg.*, labels.n_source_labels, labels.source_labels, price.median_price
FROM agg
JOIN labels USING (item_category)
LEFT JOIN price USING (item_category)
ORDER BY viewing_sessions DESC
"""

category_raw = client.query(category_sql).to_dataframe()

# Data-quality check before ranking anything: how much of the catalogue is untagged?
_views = category_raw["viewing_sessions"].sum()
_notset = category_raw.loc[category_raw["item_category"] == "(not set)", "viewing_sessions"].sum()
print(f"Raw labels: {int(category_raw['n_source_labels'].sum())}  →  categories after normalising: {len(category_raw)}")
print(f"'(not set)' share of category views: {_notset/_views*100:.1f}%")
print("Merged categories — check none of these merges two genuinely different things:")
display(category_raw[category_raw["n_source_labels"] > 1][["item_category", "n_source_labels", "source_labels", "viewing_sessions"]])

Raw labels: 74  →  categories after normalising: 47
'(not set)' share of category views: 5.0%
Merged categories — check none of these merges two genuinely different things:


,item_category,n_source_labels,source_labels,viewing_sessions
0,Men's / Unisex,2,Home/Apparel/Men's / Unisex/ · Men's / Unisex,13672
1,Sale,2,Home/Sale/ · Sale,9763
2,Apparel,2,Apparel · Home/Apparel/,6548
3,Hats,2,Hats · Home/Apparel/Hats/,6226
4,Drinkware,2,Drinkware · Lifestyle/Drinkware/,5382
5,New,2,Home/New/ · New,5335
7,Bags,2,Bags · Lifestyle/Bags/,4796
8,Women's,2,Home/Apparel/Women's/ · Women's,4778
9,YouTube,2,Home/Shop by Brand/YouTube/ · YouTube,4754
10,Small Goods,2,Lifestyle/Small Goods/ · Small Goods,3951


In [18]:
# ── Section 7d: Category ranking + chart ─────────────────────────────────────
MIN_CATEGORY_VIEWS = 500   # below this the rate is too noisy to rank

category_df = category_raw[
    (category_raw["item_category"] != "(not set)")
    & (category_raw["viewing_sessions"] >= MIN_CATEGORY_VIEWS)
].copy()
category_df["atc_pct"] = category_df["carting_sessions"] / category_df["viewing_sessions"] * 100

# Site average: pooled (volume-weighted) rate across ranked categories — internal, not an external benchmark.
# Shared with generate_dashboard.py (metrics.py, unit-tested in tests/).
from metrics import site_average, flag_categories
category_benchmark = site_average(category_df["carting_sessions"], category_df["viewing_sessions"])

# E6 — "below" only when the whole Wilson 95% interval is under the site average ("above"
# when wholly over it, else "neutral"); carts at stake count flagged categories only.
category_df = flag_categories(category_df, category_benchmark)
print(category_df["status"].value_counts().to_string())
category_df = category_df.sort_values("atc_pct").reset_index(drop=True)

print(f"Ranked categories: {len(category_df)}  |  site average seen → carted: {category_benchmark:.1f}%")
display(category_df.sort_values("carts_at_stake", ascending=False)
        .drop(columns=["source_labels"])
        .style.format({"atc_pct": "{:.1f}%", "viewing_sessions": "{:,}", "ci_lo": "{:.1f}%",
                       "ci_hi": "{:.1f}%", "carting_sessions": "{:,}", "carts_at_stake": "{:,}"}))

# Worst rate at the top; colour = significance flag (orange below, blue above, grey ≈ average).
fig_cat = go.Figure(go.Bar(
    y=category_df["item_category"], x=category_df["atc_pct"].round(1),
    orientation="h",
    text=[f"{v:.1f}%" for v in category_df["atc_pct"]], textposition="outside",
    marker_color=[{"below": "#ea580c", "above": "#4C78A8"}.get(st, "#cbd5e1") for st in category_df["status"]],
    error_x=dict(type="data", symmetric=False,
                 array=(category_df["ci_hi"] - category_df["atc_pct"]).tolist(),
                 arrayminus=(category_df["atc_pct"] - category_df["ci_lo"]).tolist(),
                 color="#6b7280", thickness=1, width=3),
))
fig_cat.add_vline(x=category_benchmark, line_dash="dash", line_color="#6b7280",
                  annotation_text=f"site average {category_benchmark:.1f}%", annotation_position="top")
fig_cat.update_layout(
    title="Seen → Carted Rate by Product Category",
    height=max(320, 30 * len(category_df) + 100),
    yaxis=dict(autorange="reversed"),
    xaxis_title="% of sessions that saw the category and carted an item from it",
    showlegend=False,
)
fig_cat.show()

# Price as a confounder: does a category's median listed price track its rate?
from scipy.stats import spearmanr
from stats_utils import format_p
_pr = category_df.dropna(subset=["median_price"])
category_price_rho, category_price_p = (float(x) for x in spearmanr(_pr["median_price"], _pr["atc_pct"]))
print(f"Spearman(median price, seen → carted) = {category_price_rho:+.2f} ({format_p(category_price_p)}), "
      f"n = {len(_pr)} — " + ("price does not explain the ranking on its own (not ruled out either)"
                             if category_price_p >= 0.05 else "price tracks the rate: part of each gap may be price"))

top_targets = (category_df.sort_values("carts_at_stake", ascending=False)
               .query("carts_at_stake > 0").head(3))
print("Start the merchandising audit with:",
      ", ".join(f"{r.item_category} ({r.carts_at_stake:,} carts at stake)" for r in top_targets.itertuples()))

status
below      12
above       8
neutral     3
Ranked categories: 23  |  site average seen → carted: 19.3%


,item_category,viewing_sessions,carting_sessions,n_source_labels,median_price,atc_pct,ci_lo,ci_hi,status,carts_at_stake
3,Apparel,"6,548",920,2,30.000000,14.1%,13.2%,14.9%,below,345
2,Bags,"4,796",654,2,12.000000,13.6%,12.7%,14.6%,below,273
11,Sale,"9,763","1,659",2,22.000000,17.0%,16.3%,17.8%,below,228
1,Eco-Friendly,"3,725",495,2,24.000000,13.3%,12.2%,14.4%,below,225
0,Google,"3,234",420,2,20.000000,13.0%,11.9%,14.2%,below,205
6,Women's,"4,778",721,2,29.000000,15.1%,14.1%,16.1%,below,202
9,YouTube,"4,754",741,2,16.000000,15.6%,14.6%,16.6%,below,178
8,Small Goods,"3,951",612,2,10.000000,15.5%,14.4%,16.7%,below,152
4,Lifestyle,"2,755",388,2,13.000000,14.1%,12.8%,15.4%,below,144
10,New,"5,335",889,2,25.000000,16.7%,15.7%,17.7%,below,142


Spearman(median price, seen → carted) = -0.33 (p = 0.123), n = 23 — price does not explain the ranking on its own (not ruled out either)
Start the merchandising audit with: Apparel (345 carts at stake), Bags (273 carts at stake), Sale (228 carts at stake)


## 8. Weekly cohort retention
Groups users by the week of their first `session_start`, then tracks what fraction
return in each subsequent week across the full Nov 2020–Jan 2021 window.

Two boundary effects are handled before any average is taken:

- **Right-censoring.** A cohort's week N is only observable if that whole week falls
  inside the data window. Unobservable cells are left **blank, not 0%** — a user who
  joined in the final week has not had the chance to return yet.
- **Partial first cohort.** Cohorts are Monday-anchored but the window opens on a
  Sunday, so the first bucket holds a single day of users. Its small base makes the
  rate noisy, so it is **excluded from the averages**.

In [19]:
# ── Section 8: Weekly cohort retention (BigQuery) ───────────────────────────
# Cohort = calendar week (Monday) of each user's first session_start.
# Retention = distinct users with session_start in week N after cohort week.
cohort_sql = f"""
WITH first_sessions AS (
  SELECT
    user_pseudo_id,
    DATE_TRUNC(PARSE_DATE('%Y%m%d', MIN(event_date)), WEEK(MONDAY)) AS cohort_week
  FROM `{DATASET}.events_*`
  WHERE _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name = 'session_start'
  GROUP BY user_pseudo_id
),
activity AS (
  SELECT DISTINCT
    user_pseudo_id,
    DATE_TRUNC(PARSE_DATE('%Y%m%d', event_date), WEEK(MONDAY)) AS activity_week
  FROM `{DATASET}.events_*`
  WHERE _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name = 'session_start'
)
SELECT
  f.cohort_week,
  DATE_DIFF(a.activity_week, f.cohort_week, WEEK) AS weeks_since_cohort,
  COUNT(DISTINCT a.user_pseudo_id)                 AS active_users
FROM first_sessions f
JOIN activity a USING (user_pseudo_id)
WHERE a.activity_week >= f.cohort_week
GROUP BY f.cohort_week, weeks_since_cohort
ORDER BY f.cohort_week, weeks_since_cohort
"""

cohort_df = client.query(cohort_sql).to_dataframe()
print(f"Cohort data rows: {len(cohort_df)}")
display(cohort_df.head(15))


Cohort data rows: 105


,cohort_week,weeks_since_cohort,active_users
0,2020-10-26,0,2339
1,2020-10-26,1,251
2,2020-10-26,2,82
3,2020-10-26,3,56
4,2020-10-26,4,31
5,2020-10-26,5,24
6,2020-10-26,6,22
7,2020-10-26,7,5
8,2020-10-26,8,14
9,2020-10-26,9,5


In [20]:
# ── Section 8b: Retention heatmap — censoring + partial-cohort guards ─────
# Same logic as generate_dashboard.py, so notebook and dashboard agree.
import datetime as _dt

cohort_pivot = cohort_df.pivot(index="cohort_week", columns="weeks_since_cohort",
                               values="active_users")

# Censoring guard: week N of a cohort is observed only if the whole week fits in
# the window. Unobserved → NaN (blank); observed but nobody returned → real 0.
_start = _dt.date(int(DATE_START[:4]), int(DATE_START[4:6]), int(DATE_START[6:8]))
_end   = _dt.date(int(DATE_END[:4]),   int(DATE_END[4:6]),   int(DATE_END[6:8]))
_cohort_dates = [pd.Timestamp(d).date() for d in cohort_pivot.index]
_observed = pd.DataFrame(
    [[(cd + _dt.timedelta(days=7 * int(k) + 6)) <= _end for k in cohort_pivot.columns]
     for cd in _cohort_dates],
    index=cohort_pivot.index, columns=cohort_pivot.columns,
)
cohort_pivot = cohort_pivot.where(_observed)
cohort_pivot = cohort_pivot.mask(_observed & cohort_pivot.isna(), 0)
censored = int((~_observed).sum().sum())
print(f"Censored cells (blank, not 0%): {censored} of {_observed.size} "
      f"({censored / _observed.size * 100:.0f}%)")

cohort_pct = cohort_pivot.div(cohort_pivot[0], axis=0).multiply(100).round(1)

# Partial-first-cohort guard: Monday-anchored buckets + a window that opens
# mid-week → the first bucket is a partial week with a much smaller base.
partial_first = _cohort_dates[0] < _start
if partial_first:
    print(f"Partial first cohort {_cohort_dates[0]}: {int(cohort_pivot[0].iloc[0]):,} users "
          f"vs {int(cohort_pivot[0].iloc[1]):,} in the first full cohort — excluded from averages")

cohort_pct.index   = cohort_pct.index.astype(str)
cohort_pct.columns = [f"Week {c}" for c in cohort_pct.columns]
cohort_pct_full    = cohort_pct.iloc[1:] if partial_first else cohort_pct

print("Retention matrix (% of cohort returning; blank = not yet observable):")
display(cohort_pct)

z   = cohort_pct.values.tolist()
txt = [["" if pd.isna(v) else f"{v:.0f}%" for v in row] for row in z]

# Blues runs light → dark as the value rises, so darker = better retention.
# zmax=12 keeps the 0–12% band readable; Week 0 is 100% by definition.
fig_ret = go.Figure(go.Heatmap(
    z=z, x=cohort_pct.columns.tolist(), y=cohort_pct.index.tolist(),
    text=txt, texttemplate="%{text}",
    colorscale="Blues", zmin=0, zmax=12, hoverongaps=False,
))
fig_ret.update_layout(
    title="Weekly Cohort Retention (% returning)",
    xaxis_title="Weeks Since First Session",
    yaxis_title="Cohort Week",
    height=500,
    yaxis_autorange="reversed",
)
fig_ret.show()

# Headline KPIs: full cohorts only; censored (NaN) cells are skipped by .mean().
if "Week 1" in cohort_pct_full.columns:
    # Week-1 return = any session in the next Monday-anchored calendar week (1–13 days), not D7.
    print(f"\nAvg Week-1 return (full cohorts): {cohort_pct_full['Week 1'].mean():.1f}%")
if "Week 2" in cohort_pct_full.columns:
    print(f"Avg Week-2 return (full cohorts): {cohort_pct_full['Week 2'].mean():.1f}%")

Censored cells (blank, not 0%): 91 of 196 (46%)
Partial first cohort 2020-10-26: 2,339 users vs 19,194 in the first full cohort — excluded from averages
Retention matrix (% of cohort returning; blank = not yet observable):


,Week 0,Week 1,Week 2,Week 3,Week 4,Week 5,Week 6,Week 7,Week 8,Week 9,Week 10,Week 11,Week 12,Week 13
cohort_week,,,,,,,,,,,,,,
2020-10-26,100.0,10.7,3.5,2.4,1.3,1.0,0.9,0.2,0.6,0.2,0.4,0.3,0.3,0.3
2020-11-02,100.0,6.2,3.4,2.3,2.3,1.9,1.3,0.6,0.4,0.7,0.5,0.4,0.4,<NA>
2020-11-09,100.0,6.7,3.0,2.6,2.4,1.5,0.8,0.4,0.6,0.5,0.5,0.4,<NA>,<NA>
2020-11-16,100.0,5.7,3.3,2.4,1.8,0.7,0.4,0.5,0.5,0.4,0.3,<NA>,<NA>,<NA>
2020-11-23,100.0,5.0,2.5,1.5,0.7,0.4,0.5,0.5,0.4,0.3,<NA>,<NA>,<NA>,<NA>
2020-11-30,100.0,5.3,2.2,0.9,0.6,0.7,0.5,0.4,0.4,<NA>,<NA>,<NA>,<NA>,<NA>
2020-12-07,100.0,3.3,0.9,0.7,1.0,0.6,0.4,0.4,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2020-12-14,100.0,2.9,0.9,1.2,0.9,0.6,0.4,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2020-12-21,100.0,2.5,1.1,1.0,0.6,0.4,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>



Avg Week-1 return (full cohorts): 4.2%
Avg Week-2 return (full cohorts): 1.9%


## 8c. New vs returning sessions (E3)

A session is **new** when GA4's `ga_session_number` = 1 (the user's first session ever on that
cookie — counted from first visit, not from the start of this window), otherwise **returning**.
Computed on the same per-session scan and all-sessions basis as Section 5e.

**Read with care:** returning sessions convert better partly because users who come back
unprompted were keener to begin with. The returning rate is therefore an *upper bound* on what
re-engaged users would be worth; the new-session rate is the lower bound. Recommendation 2 in
the dashboard is sized between the two.

In [21]:
# ── Section 8c: New vs returning sessions (E3) ─────────────────────────────
from stats_utils import wilson_ci, chi_square, format_p

vis = dq_raw[dq_raw["cut"] == "visitor"].set_index("visitor")
visitor_df = pd.DataFrame({
    "sessions":                 vis["sessions"],
    "saw_product_pct":          vis["view_item"] / vis["sessions"] * 100,
    "saw_to_cart_pct":          vis["add_to_cart"] / vis["view_item"] * 100,
    "cart_to_checkout_pct":     vis["begin_checkout"] / vis["add_to_cart"] * 100,
    "checkout_to_purchase_pct": vis["purchase"] / vis["begin_checkout"] * 100,
    "session_to_purchase_pct":  vis["purchase"] / vis["sessions"] * 100,
    "revenue_per_session":      vis["revenue"] / vis["sessions"],
})
_ci = [wilson_ci(p, n) for p, n in zip(vis["purchase"], vis["sessions"])]
visitor_df["purchase_ci_lo"], visitor_df["purchase_ci_hi"] = [c[0] for c in _ci], [c[1] for c in _ci]
display(visitor_df.round(2))

visitor_test = chi_square(vis.loc[["new", "returning"], "purchase"], vis.loc[["new", "returning"], "sessions"])
ratio = visitor_df.loc["returning", "session_to_purchase_pct"] / visitor_df.loc["new", "session_to_purchase_pct"]
print(f"Returning vs new purchase rate: {ratio:.1f}×  ({format_p(visitor_test['p'])}, "
      f"Cramér's V = {visitor_test['cramers_v']:.3f})")

# Recommendation 2 in revenue: users the 8% Week-1 planning target would add, one session each,
# valued between a new and a returning session (selection bias makes the latter an upper bound).
WK1_TARGET = 8.0
_full = cohort_pivot.iloc[1:] if partial_first else cohort_pivot
_full = _full[_full[1].notna()]
extra_users = max(0, round(_full[0].sum() * WK1_TARGET / 100 - _full[1].sum()))
rec2_low  = extra_users * visitor_df.loc["new", "revenue_per_session"]
rec2_high = extra_users * visitor_df.loc["returning", "revenue_per_session"]
print(f"Week-1 return at {WK1_TARGET:.0f}% (planning assumption) → ~{extra_users:,} extra returning users "
      f"→ ${rec2_low:,.0f}–${rec2_high:,.0f} over the window")

,sessions,saw_product_pct,saw_to_cart_pct,cart_to_checkout_pct,checkout_to_purchase_pct,session_to_purchase_pct,revenue_per_session,purchase_ci_lo,purchase_ci_hi
visitor,,,,,,,,,
new,257400,19.8,17.03,29.78,41.56,0.42,0.27,0.39,0.44
returning,97457,24.94,25.48,42.32,62.84,1.69,1.27,1.61,1.77


Returning vs new purchase rate: 4.1×  (p < 0.001, Cramér's V = 0.065)
Week-1 return at 8% (planning assumption) → ~9,444 extra returning users → $2,563–$11,999 over the window


## 9. Summary & hypotheses

No external benchmarks are used: none could be sourced for this store type, so targets in the
dashboard are labelled planning assumptions and comparisons are internal (site average).

| Metric | What to look for | Hypothesis if it is the weak step |
|---|---|---|
| **Session → View Products** | Largest absolute loss? Check `(not set)` share first (5e) | Landing page / homepage routing to products |
| **View Products → Add to Cart** | Spread across categories (7c), not the pooled rate | Listing presentation, price, stock — check price as a confounder |
| **Add to Cart → Checkout** | Cart abandonment — the main post-intent loss | Shipping-cost surprise, forced account creation, no cart recovery |
| **Checkout → Purchase** | Drop inside checkout | Payment or form friction |
| **Shoppers funnel ATC rate** | The step's own rate | Not comparable to the all-sessions ATC rate (different denominator) |
| **AOV / revenue per purchase** | Purchases without revenue (Section 4) | Instrumentation gap, not behaviour |
| **Time-to-convert** | Identical-timestamp share (Section 6) | Tracking signal, not shopper speed |
| **Week-1 return** | Trend across cohorts, not just the average | Seasonal traffic mix, no re-engagement |
| **Mobile vs desktop** | Gap in purchase rate | If small, device is not the lever |

---

### Next steps

**Point at real data:** swap `bigquery-public-data.ga4_obfuscated_sample_ecommerce` for your own
GA4 BigQuery export dataset. The SQL patterns are identical.

**Add segments:** channel (Section 5c) and product category (Section 7c) are done.
Next candidates: `geo.country`, or a `new_vs_returning` flag (`ga_session_number = 1` = new user).
When segmenting a *session* funnel, use session-level attributes — `traffic_source.*`
is first-touch and user-level (see the grain note in 5c).

**Statistical rigour:** for device or period comparisons, add a proportion Z-test
to confirm differences are real and not sampling noise — especially important
when tablet N is small.

## 10. AI-Powered Insight Generation

Pass all analysis results to an LLM to produce a **product-decision-ready brief** —
the kind of output a Product Analyst shares with a squad after completing an analysis.

This section demonstrates three things that map directly to modern product analytics roles:

- **Prompt engineering** — a structured system prompt with an explicit output schema
- **Contextualised LLM calls** — serialising DataFrame results as structured JSON context
- **AI-assisted insight workflows** — replacing manual write-up time with a reproducible, auditable step
- **Guardrails** — the computed verdicts are passed explicitly and the prompt forbids contradicting them;
  every hypothesis must name a confounder. The brief is stamped with the context version, and the
  dashboard flags a brief written from an older context.

> **API key (optional).** Keys are read from environment variables only — never paste
> one into the notebook, because outputs are committed.
> - **Anthropic Claude** (paid): key from [console.anthropic.com](https://console.anthropic.com) → `export ANTHROPIC_API_KEY=...` → used by **10d**
> - **Google Gemini** (free): key from [aistudio.google.com](https://aistudio.google.com) → `export GEMINI_API_KEY=...` → used by **10e**, only if 10d produced no brief
> - **Neither set:** 10d–10f are skipped. The committed `dashboard.html` keeps the brief it already embeds.
>
> 10f writes `product_brief_ga4_funnel.md` and, next to it, `product_brief_context.json` —
> the exact prompts and context the brief was written from, so every claim can be traced
> to a computed number. Both are generated output and gitignored;
> `generate_dashboard.py` embeds it when present.

In [22]:
# ── Section 10a: Install SDK(s) ─────────────────────────────────────────────
# Install the Anthropic SDK (used in 10d). If you prefer the free Gemini
# alternative, also install google-genai (see Section 10e).
%pip install -q anthropic
%pip install -q google-genai


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [23]:
# ── Section 10b: API key ───────────────────────────────────────────────────
import os

# Recommended: set in your shell before launching Jupyter
#   export ANTHROPIC_API_KEY="sk-ant-..."
# Fallback: paste key directly here for local use only (never commit).
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "YOUR_API_KEY_HERE")

if ANTHROPIC_API_KEY == "YOUR_API_KEY_HERE":
    print("Set ANTHROPIC_API_KEY before running Section 10.")
else:
    print("API key loaded.")


Set ANTHROPIC_API_KEY before running Section 10.


In [24]:
# ── Section 10c: Compile analysis context for the LLM ─────────────────────
import json

def _f(val):
    """Safely convert a pandas/numpy scalar to a rounded Python float (NaN → None)."""
    try:
        v = float(val)
        return None if v != v else round(v, 2)  # NaN → None
    except (TypeError, ValueError):
        return None

# Version of this context's structure. 10f stamps it on the brief and
# generate_dashboard.py flags a brief from an older version as out of date.
# Must match BRIEF_CONTEXT_VERSION in generate_dashboard.py.
BRIEF_CONTEXT_VERSION = 3

# ctx: structured dict passed as JSON in the user prompt to the LLM.
ctx = {}

# Session funnel (Section 3) — requires funnel_df from prior cells.
try:
    ctx["session_funnel"] = [
        {
            "step": r["step"],
            "sessions": int(r["sessions"]),
            "step_conversion_pct": _f(r["step_cvr"]),
            "overall_conversion_pct": _f(r["overall_cvr"]),
            "drop_off_pct": _f(r["drop_off"]),
        }
        for _, r in funnel_df.iterrows()
    ]
except NameError:
    print("funnel_df not found — run Section 3 first")

# Revenue summary (Section 4).
try:
    rv = revenue_df.iloc[0]
    ctx["revenue"] = {
        "funnel_purchases": int(rv["purchasing_sessions"]),
        "orders_with_revenue_recorded": int(rv["orders_with_revenue"]),
        "total_revenue_usd": _f(rv["total_revenue"]),
        "avg_order_value_usd": _f(rv["avg_order_value"]),
        "revenue_per_funnel_purchase_usd": _f(rv["total_revenue"] / rv["purchasing_sessions"]),
        "median_order_value_usd": _f(rv["median_order_value"]),
        "all_purchasing_sessions_incl_non_sequential": int(rv["all_purchasing_sessions"]),
        "all_recorded_revenue_usd": _f(rv["all_revenue"]),
        # Counts stated explicitly: a note that only describes them invites the
        # model to estimate a number of its own.
        "purchasing_sessions_outside_sequential_funnel": int(rv["all_purchasing_sessions"] - rv["purchasing_sessions"]),
        "purchasing_sessions_without_add_to_cart_in_session": int(rv["purchases_without_cart"]),
        "note": ("AOV averages only orders with revenue recorded; use revenue_per_funnel_purchase "
                 "for sizing. Funnel revenue covers sequential sessions only — most other purchasing "
                 "sessions have no add_to_cart in the same session (cart filled in an earlier visit)."),
    }
except NameError:
    print("revenue_df not found — run Section 4 first")

# Device breakdown at purchase step (Section 5).
try:
    purchase_rows = device_long[device_long["step_col"] == "step5_purchase"]
    ctx["purchase_rate_by_device"] = [
        {
            "device": r["device_category"],
            "sessions": int(r["sessions"]),
            "pct_of_device_sessions": _f(r["pct"]),
        }
        for _, r in purchase_rows.iterrows()
    ]
except NameError:
    print("device_long not found — run Section 5 first")

# Median minutes between funnel steps (Section 6) + tracking check.
try:
    ctx["median_minutes_between_steps_converters_only"] = {
        "session_to_view_products": _f(ttc_df["p50_s_to_v"].iloc[0]),
        "view_products_to_add_to_cart": _f(ttc_df["p50_v_to_c"].iloc[0]),
        "add_to_cart_to_checkout": _f(ttc_df["p50_c_to_k"].iloc[0]),
        "checkout_to_purchase": _f(ttc_df["p50_k_to_p"].iloc[0]),
        "total_session_to_purchase": _f(ttc_df["p50_total"].iloc[0]),
        "carting_sessions_with_identical_view_and_cart_timestamp_pct": _f(cart_same_ts_pct),
        "note": ("Describes converting sessions only — says nothing about why carts are abandoned. "
                 "Identical view/cart timestamps are a tracking signal (listing quick-add), not "
                 "evidence that shoppers are decisive."),
    }
except NameError:
    print("ttc_df not found — run Section 6 first")

# Shoppers vs full funnel add-to-cart comparison (Section 7).
try:
    full_cart = funnel_df.loc[funnel_df["step"] == "3. Add to Cart", "overall_cvr"].values
    shop_cart = shoppers_df.loc[shoppers_df["step"] == "2. Add to Cart", "step_cvr"].values
    ctx["shoppers_funnel"] = {
        "note": ("Funnel starting at view_item — removes sessions that never saw a product. "
                 "Both rates share a numerator; their ratio only restates the discovery loss and "
                 "says nothing about product presentation."),
        "add_to_cart_rate_from_all_sessions_pct": _f(full_cart[0]) if len(full_cart) else None,
        "add_to_cart_rate_from_product_viewers_pct": _f(shop_cart[0]) if len(shop_cart) else None,
    }
except NameError:
    print("shoppers_df not found — run Section 7 first")

# Cohort retention averages (Section 8).
try:
    # Full cohorts only (partial first cohort excluded); censored cells are NaN and skipped.
    wk1 = cohort_pct_full["Week 1"].mean() if "Week 1" in cohort_pct_full.columns else None
    wk2 = cohort_pct_full["Week 2"].mean() if "Week 2" in cohort_pct_full.columns else None
    ctx["cohort_retention"] = {
        "avg_week1_return_pct": _f(wk1),
        "avg_week2_return_pct": _f(wk2),
        "definition": ("Week-1 return = share of a weekly cohort with any session in the next "
                       "Monday-anchored calendar week (a 1-13 day window) — not D7 retention."),
        "planning_target_pct": 8.0,
        "planning_target_note": "an analyst assumption, not an external benchmark",
        "note": ("% of users with any session in week N after their acquisition week; "
                 "full cohorts only, right-censored weeks excluded. Measures return visits, "
                 "not purchases: it cannot show whether buyers decided in one session or "
                 "researched across sessions (cross-device and cleared cookies also look like "
                 "non-returns)."),
    }
except NameError:
    print("cohort_pct_full not found — run Section 8 first")

# Acquisition channel split (Section 5c).
try:
    ctx["funnel_by_channel"] = {
        "channel_definition": channel_dim,
        "view_item_spread_pp": _f(spread),
        "decision_rule": ("10pp spread in view_item_pct = traffic-quality problem; 7-10pp = borderline; "
                          "below 7pp = site problem. The 10pp threshold is an analyst-set assumption."),
        "verdict": verdict,
        "note": ("low_purchase_volume=True means fewer than 30 purchases: purchase_pct and "
                 "revenue_per_session are too noisy to rank or act on. view_item_pct rests on "
                 "session counts and is reliable. '(not set)' sessions carry no session-level "
                 "medium and are excluded from the spread; they are short sessions missing the tag, "
                 "not a channel (see data_quality). '(none)' = direct traffic."),
        "rows": [
            {
                "channel": r["channel"],
                "sessions": int(r["sessions"]),
                "view_item_pct": _f(r["view_pct"]),
                "purchase_pct": _f(r["purchase_pct"]),
                "revenue_per_session_usd": _f(r["rev_per_session"]),
                "low_purchase_volume": bool(r["low_n"]),
            }
            for _, r in channel_df.iterrows()
        ],
    }
except NameError:
    print("channel_df not found — run Section 5c first")

# Product category view → cart (Section 7c).
try:
    ctx["category_view_to_cart"] = {
        "site_average_pct": _f(category_benchmark),
        "metric_definition": ("seen_to_carted_pct = of sessions in which a product from this category "
                              "appeared in a viewed product list (view_item is list-level in this export), "
                              "the % that then carted an item from this category (the carted item only, "
                              "quantity > 0). It is evidence about how the category's products convert "
                              "from listings, not about product detail pages."),
        "note": ("carts_at_stake = extra carting sessions if the category converted at the site average "
                 "(pooled rate across ranked categories — internal, not an external benchmark). Counted only "
                 "for categories whose whole 95% interval is below the site average (vs_site_average='below'); "
                 "'neutral' categories cannot be distinguished from average and must not be called "
                 "underperforming. About one flag in 20 may be chance."),
        "rows": [
            {
                "category": r["item_category"],
                "viewing_sessions": int(r["viewing_sessions"]),
                "seen_to_carted_pct": _f(r["atc_pct"]),
                "ci95_pct": [_f(r["ci_lo"]), _f(r["ci_hi"])],
                "vs_site_average": r["status"],
                "median_listed_price_usd": _f(r["median_price"]),
                "carts_at_stake": int(r["carts_at_stake"]),
            }
            for _, r in category_df.sort_values("carts_at_stake", ascending=False).iterrows()
        ],
    }
except NameError:
    print("category_df not found — run Section 7c first")

# Post-intent step losses (Section 3) — named explicitly so they are not missed.
try:
    _s = funnel_df.set_index("step")["sessions"]
    ctx["post_intent_losses"] = {
        "cart_to_checkout_pct": _f(_s["4. Begin Checkout"] / _s["3. Add to Cart"] * 100),
        "cart_abandonment_pct": _f(100 - _s["4. Begin Checkout"] / _s["3. Add to Cart"] * 100),
        "checkout_to_purchase_pct": _f(_s["5. Purchase"] / _s["4. Begin Checkout"] * 100),
    }
except (NameError, KeyError):
    print("funnel_df not found — run Section 3 first")

# Data quality (Section 5e).
try:
    ctx["data_quality"] = {
        "unattributed_sessions_pct": _f(ua_share),
        "unattributed_saw_product_pct": _f(ua_view_pct),
        "attributed_saw_product_pct": _f(at_view_pct),
        "share_of_discovery_loss_in_unattributed_pct": _f(ua_lost_share),
        "unattributed_interpretation": ("short sessions missing the medium tag, not bots and not a "
                                        "channel: at matched session length their view rate is close "
                                        "to attributed sessions. All sessions remain the primary view."),
        "view_item_events_with_multiple_products_pct": _f(vi_multi_pct),
        "view_item_interpretation": ("view_item is a list impression here, so 'saw products' — not "
                                     "'viewed a product page'. Do not make product-detail-page claims."),
    }
except NameError:
    print("dq_raw not found — run Section 5e first")

# New vs returning sessions (Section 8c, E3).
try:
    ctx["new_vs_returning"] = {
        "definition": "new = ga_session_number 1 (first session ever for that cookie); otherwise returning",
        "rows": {k: {c: _f(v) for c, v in row.items()} for k, row in visitor_df.iterrows()},
        "returning_to_new_purchase_rate_ratio": _f(ratio),
        "chi_square_p": format_p(visitor_test["p"]),
        "rec2_extra_returning_users_at_8pct_target": int(extra_users),
        "rec2_revenue_range_usd": [_f(rec2_low), _f(rec2_high)],
        "note": ("Returning sessions convert better partly through self-selection (keener users come "
                 "back unprompted), so the returning rate is an upper bound for re-engaged users. Do not "
                 "claim that bringing users back will multiply their conversion by the ratio."),
    }
except NameError:
    print("visitor_df not found — run Section 8c first")

# Statistical significance (Section 5f, E6).
try:
    ctx["significance"] = {
        "device_purchase_chi_square": {"p": format_p(device_test["p"]), "cramers_v": round(device_test["cramers_v"], 4),
                                       "verdict": "significant" if device_test["p"] < 0.05 else "not significant"},
        "device_purchase_ci95_pct": {r["device_category"]: [_f(r["ci_lo"]), _f(r["ci_hi"])]
                                     for _, r in dev_sig.iterrows()},
        "channel_saw_product_chi_square": {"p": format_p(channel_test["p"]), "cramers_v": round(channel_test["cramers_v"], 3)},
        "note": ("The channel spread is statistically significant (large samples) but the site-vs-traffic "
                 "verdict is about practical size — the 10pp threshold — not significance."),
    }
except NameError:
    print("device_test not found — run Section 5f first")

# Computed verdicts (all sections). Built from this run's numbers, so the brief
# can be told explicitly what it must not contradict.
from stats_utils import format_p
try:
    _s = funnel_df.set_index("step")["sessions"]
    _drops = (_s - _s.shift(-1)).dropna()
    _from = _drops.idxmax(); _to = _s.index[list(_s.index).index(_from) + 1]
    _abandon = 100 - _s["4. Begin Checkout"] / _s["3. Add to Cart"] * 100
    _c2p = _s["5. Purchase"] / _s["4. Begin Checkout"] * 100
    _below = category_df[category_df["status"] == "below"].sort_values("carts_at_stake", ascending=False)
    _top = _below.head(3)
    _wk1 = cohort_pct_full["Week 1"].mean()
    ctx["computed_verdicts"] = {
        "primary_leak": (f"Discovery. The largest absolute loss is between {_from} and {_to} "
                         f"({int(_drops.max()):,} sessions); {_s.iloc[1] / _s.iloc[0] * 100:.1f}% of sessions saw any product."),
        "saw_to_cart": (f"{_s['3. Add to Cart'] / _s.iloc[1] * 100:.1f}% of sessions that saw products added one to cart. "
                        "This is a separate problem from discovery and it depends on the category."),
        "categories": (f"{len(_below)} of {len(category_df)} ranked categories sit below the "
                       f"{category_benchmark:.1f}% site average with 95% confidence. The top "
                       f"{len(_top)} by carts at stake are {', '.join(_top['item_category'])} "
                       f"({int(_top['carts_at_stake'].sum()):,} carts at stake together, of "
                       f"{int(_below['carts_at_stake'].sum()):,} across all flagged categories). "
                       "Categories marked 'neutral' are not underperforming."),
        "cart_abandonment": (f"{_abandon:.0f}% of carting sessions never begin checkout"
                             + (" — the largest loss after a shopper shows purchase intent." if _abandon > 100 - _c2p else ".")),
        "checkout": (f"{_c2p:.1f}% of sessions that begin checkout complete a purchase. Nothing in the data "
                     "compares this with another store or period: do not call checkout efficient, lean or broken."),
        "channel": (f"{verdict} (spread {spread:.1f}pp against the 10pp analyst-set threshold). The spread is "
                    f"statistically significant ({format_p(channel_test['p'])}) but small; the verdict rests on "
                    "practical size. Do not attribute the discovery loss to traffic quality."),
        "device": (f"Device does not change the purchase rate ({format_p(device_test['p'])}); intervals overlap."
                   if device_test["p"] >= 0.05 else
                   f"Device differences are significant ({format_p(device_test['p'])}) but negligible in size."),
        "retention": (f"Average Week-1 return is {_wk1:.1f}% against an 8% planning target — an analyst "
                      "assumption, not an industry benchmark."),
        "new_vs_returning": (f"Returning sessions buy at {ratio:.1f}x the new-session rate; part of that is "
                             "self-selection, so it is not the effect of bringing someone back."),
        "unattributed": (f"'(not set)' sessions ({ua_share:.1f}% of sessions, {ua_lost_share:.0f}% of the discovery "
                         "loss) are short sessions missing the medium tag — not bots, not a channel, not broken landing pages."),
    }
except NameError as e:
    print(f"computed_verdicts incomplete — run all sections first ({e})")

# Known confounders: what else could produce each pattern. The brief must state
# the relevant one for every hypothesis.
try:
    ctx["known_confounders"] = [
        {"applies_to": "category seen → carted rates",
         "confounder": "price",
         "evidence": (f"Rank correlation of category median listed price with the rate: {category_price_rho:+.2f} "
                      f"({format_p(category_price_p)}). "
                      + ("Too weak to explain the ranking on its own, but not ruled out — cheap items are an easier yes."
                         if category_price_p >= 0.05 else "Cheaper categories convert better; part of each gap may be price."))},
        {"applies_to": "category seen → carted rates",
         "confounder": "listing composition",
         "evidence": ("'Saw' means a product appeared in a viewed list. Categories that appear in broad, mixed lists "
                      "are 'seen' by less interested sessions than categories with their own focused listing.")},
        {"applies_to": "new vs returning",
         "confounder": "self-selection",
         "evidence": "Users who return unprompted were keener to begin with."},
        {"applies_to": "discovery loss",
         "confounder": "very short, untagged sessions",
         "evidence": f"{ua_lost_share:.0f}% of the loss is in short sessions without a channel tag; their intent is unknown."},
        {"applies_to": "all purchase and revenue metrics",
         "confounder": "session-scoped sequential funnel",
         "evidence": "Purchases whose cart was filled in an earlier session are not counted."},
        {"applies_to": "everything",
         "confounder": "obfuscated public sample",
         "evidence": "Some values are masked or deleted; findings need validation on production data."},
    ]
except NameError as e:
    print(f"known_confounders incomplete — run all sections first ({e})")

print("Context compiled — passing to LLM:")
# default=str avoids crashes if any value is still a numpy/pandas scalar
print(json.dumps(ctx, indent=2, default=str))


Context compiled — passing to LLM:
{
  "session_funnel": [
    {
      "step": "1. Session Start",
      "sessions": 354857,
      "step_conversion_pct": null,
      "overall_conversion_pct": 100.0,
      "drop_off_pct": null
    },
    {
      "step": "2. View Products",
      "sessions": 75261,
      "step_conversion_pct": 21.2,
      "overall_conversion_pct": 21.2,
      "drop_off_pct": 78.8
    },
    {
      "step": "3. Add to Cart",
      "sessions": 14870,
      "step_conversion_pct": 19.8,
      "overall_conversion_pct": 4.2,
      "drop_off_pct": 80.2
    },
    {
      "step": "4. Begin Checkout",
      "sessions": 5205,
      "step_conversion_pct": 35.0,
      "overall_conversion_pct": 1.5,
      "drop_off_pct": 65.0
    },
    {
      "step": "5. Purchase",
      "sessions": 2721,
      "step_conversion_pct": 52.3,
      "overall_conversion_pct": 0.8,
      "drop_off_pct": 47.7
    }
  ],
  "revenue": {
    "funnel_purchases": 2721,
    "orders_with_revenue_recorded": 2461,

In [25]:
# ── Section 10d: Generate product brief via Claude ─────────────────────────
import anthropic

# System prompt: fixed output schema + product-analyst tone (not data-science jargon).
SYSTEM_PROMPT = """You are a senior Product Analyst embedded in a cross-functional product squad.
Your job is to turn computed funnel results into a clear, decision-ready product brief.

Produce exactly these five sections, in this markdown format:

## Executive Summary
2-3 sentences: the most important finding and its business implication.

## Key Findings
3-5 numbered findings. Each: a bold title and the specific numbers, then one sub-bullet
"   - **So what:** ..." that a product manager can act on.

## Hypotheses
3 numbered hypotheses, each titled "**[HIGH | MEDIUM | LOW] confidence — <claim>**", with sub-bullets:
   - **Evidence:** the numbers from the data that support it
   - **Confounder:** what else could produce the same pattern (use known_confounders)

## Recommended Actions
3 numbered actions, each with sub-bullets **What:**, **Why:** and **Metric:**. Give a target only
if the data provides one, and label it a planning assumption.

## What to Measure Next
2-3 analyses that would confirm or refute the hypotheses.

Hard rules — a brief that breaks one is rejected:
1. computed_verdicts are final. Every sentence must be consistent with them. Never contradict a
   verdict, reverse it, or re-derive it differently — for example, do not attribute the
   discovery loss to traffic quality when the channel verdict says the leak is on the site.
2. Use only numbers present in the data and copy them exactly, including how many items a figure
   covers: if the data names three categories with N carts at stake, write three and N. Do not
   compute new totals.
3. State a confounder for every hypothesis. A hypothesis whose main confounder the data does not
   rule out cannot be HIGH confidence.
4. Use each metric only for what its definition says. view_item is a list-level event in this
   export: write "saw products", never "product page" or "PDP", and make no claim about product
   detail pages.
5. Comparisons are internal: say "site average", never "benchmark". Targets in the data are
   planning assumptions — say so. Never invent industry benchmarks.
6. Do not call a step healthy, efficient, lean or strong unless the data compares it with something.
7. Respect data-quality flags: never base a finding or an action on a low-volume metric;
   '(not set)' is a tagging gap, not a channel or traffic source; a statistically significant
   difference is not automatically a large one.
8. Fast conversion times and low return rates do not reveal buyer motivation; do not claim it.
9. Write for a product manager: opinionated, plain language, no statistical jargon beyond
   "significant" and "confidence interval".
10. State that the data is an obfuscated public sample and the findings need validation on real data.
11. Where useful, give the B2B SaaS equivalent (session_start → login, view_item → feature
    discovery, purchase → activation).
12. Keep each verdict's level of certainty. A verdict described as borderline, leaning or
    "the leading explanation" stays that way: never write that the data "confirms", "proves"
    or "shows" it. Hedges in computed_verdicts are part of the verdict.
13. Round to the precision a reader needs: whole dollars or $k for money (e.g. $2.6k–$12.0k,
    never cents on an estimate), one decimal for percentages and percentage points. Label as
    a planning assumption only the input that is one (e.g. the 8% target), not the estimate
    derived from it."""

# User prompt embeds compiled ctx JSON from the previous cell.
USER_PROMPT = f"""Produce a product brief from these funnel analysis results.

Dataset: {DATASET_LABEL} ({PERIOD_LABEL}, session-scoped funnel).
The methodology is directly transferable to B2B SaaS — where "purchase" maps to
a key activation event such as completing onboarding or adopting a core feature.

These analyses have ALREADY been done and their results are in the data below: the
acquisition-channel split (funnel_by_channel), the product-category split with confidence
intervals and prices (category_view_to_cart), new vs returning sessions (new_vs_returning),
significance tests (significance) and data-quality checks (data_quality). Interpret them; do not
recommend running any of them as a next step.

Start from computed_verdicts — they are final — and use known_confounders for the Confounder lines.

{json.dumps(ctx, indent=2)}"""


def generate_brief(api_key: str) -> str:
    """Call Claude with system + user prompts; return markdown brief text."""
    client = anthropic.Anthropic(api_key=api_key)
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1800,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": USER_PROMPT}],
    )
    return response.content[0].text


if ANTHROPIC_API_KEY != "YOUR_API_KEY_HERE":
    print("Calling Claude... (~10-20 seconds)\n")
    brief = generate_brief(ANTHROPIC_API_KEY)
    BRIEF_MODEL = "Claude (claude-sonnet-4-6)"
    print(brief)
else:
    brief = None
    print("Skipped — set ANTHROPIC_API_KEY first.")


Skipped — set ANTHROPIC_API_KEY first.


In [26]:
# ── Section 10e: FREE ALTERNATIVE — Google Gemini ───────────────────────────
# Runs only if 10d did not produce a brief (no Anthropic key) and a Gemini key is set.
# Setup (one-time): get a free key at https://aistudio.google.com → "Get API key",
#   export GEMINI_API_KEY="..."   (never paste the key into the notebook — outputs get committed)
# Uses the google-genai SDK; the older google-generativeai package is deprecated.
# Model IDs change often — check https://ai.google.dev/gemini-api/docs/models and
# override with export GEMINI_MODEL="..." if the default is retired.
import os

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "YOUR_GEMINI_KEY_HERE")
GEMINI_MODEL   = os.environ.get("GEMINI_MODEL", "gemini-3.5-flash")

if brief:
    print("Brief already generated in 10d — skipping Gemini.")
elif GEMINI_API_KEY == "YOUR_GEMINI_KEY_HERE":
    print("Skipped — set GEMINI_API_KEY (free) or ANTHROPIC_API_KEY to generate a brief.")
else:
    from google import genai
    from google.genai import types

    gemini_client = genai.Client(api_key=GEMINI_API_KEY)   # not `client`: that is BigQuery
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=USER_PROMPT,                               # from Section 10d
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,               # from Section 10d
            max_output_tokens=8192,   # thinking tokens count toward this on Flash models
        ),
    )
    brief = response.text
    BRIEF_MODEL = f"Gemini ({GEMINI_MODEL})"
    _finish = response.candidates[0].finish_reason if response.candidates else None
    if str(_finish).endswith("MAX_TOKENS"):
        print("WARNING: brief was cut off at max_output_tokens — raise it and re-run.\n")
    print(brief)

## Executive Summary
The primary funnel leak occurs during initial product discovery, where 78.8% of sessions drop off before seeing products (in B2B SaaS, equivalent to dropping off between login and feature discovery). The channel spread leans toward a site-level navigation issue rather than traffic quality, complemented by significant post-intent cart abandonment where 65.0% of carting sessions fail to enter checkout. Note that these findings are derived from an obfuscated public sample dataset (GA4 Nov 2020 – Jan 2021) and require validation on production data.

## Key Findings
1. **Product discovery is the largest absolute leak in the funnel.**
   Out of 354,857 total sessions, only 75,261 reach the step where users saw products (21.2% step conversion rate), resulting in a drop-off of 279,596 sessions (78.8%). In B2B SaaS terms, this represents a failure to move users from initial session start (login) to feature discovery.
   - **So what:** Acquisition channels are driving users 

In [27]:
# ── Section 10f: Display and export product brief ─────────────────────────
from IPython.display import Markdown, display
from pathlib import Path
import json
import pandas as pd

if brief:
    # Render Claude output as formatted markdown inside the notebook.
    display(Markdown(brief))

    # Persist to repo root for README, portfolio, or sharing outside Jupyter.
    header = (
        "# Product Brief — GA4 Funnel Analysis\n\n"
        f"*Generated {pd.Timestamp.now().strftime('%Y-%m-%d')} · Model: {BRIEF_MODEL} · "
        f"Context v{BRIEF_CONTEXT_VERSION} · "
        f"Session-scoped funnel · {PERIOD_LABEL} · {DATASET_MD}*\n\n"
        "---\n\n"
    )
    out = Path("product_brief_ga4_funnel.md")
    out.write_text(header + brief, encoding="utf-8")
    print(f"Brief saved to {out}")

    # Audit trail: the exact prompts and context the brief was written from, so
    # every claim in it can be traced back to a computed number. Gitignored like
    # the brief (generated output).
    audit = {
        "generated": pd.Timestamp.now().isoformat(timespec="seconds"),
        "model": BRIEF_MODEL,
        "context_version": BRIEF_CONTEXT_VERSION,
        "system_prompt": SYSTEM_PROMPT,
        "user_prompt": USER_PROMPT,   # contains the context exactly as sent
        "context": ctx,
    }
    audit_out = Path("product_brief_context.json")
    audit_out.write_text(json.dumps(audit, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    print(f"Context saved to {audit_out}")


## Executive Summary
The primary funnel leak occurs during initial product discovery, where 78.8% of sessions drop off before seeing products (in B2B SaaS, equivalent to dropping off between login and feature discovery). The channel spread leans toward a site-level navigation issue rather than traffic quality, complemented by significant post-intent cart abandonment where 65.0% of carting sessions fail to enter checkout. Note that these findings are derived from an obfuscated public sample dataset (GA4 Nov 2020 – Jan 2021) and require validation on production data.

## Key Findings
1. **Product discovery is the largest absolute leak in the funnel.**
   Out of 354,857 total sessions, only 75,261 reach the step where users saw products (21.2% step conversion rate), resulting in a drop-off of 279,596 sessions (78.8%). In B2B SaaS terms, this represents a failure to move users from initial session start (login) to feature discovery.
   - **So what:** Acquisition channels are driving users to the site, but the initial navigation or landing experience fails to surface products to the vast majority of visitors.

2. **Severe cart abandonment accounts for the largest post-intent loss.**
   Of the 14,870 sessions that added an item to cart, only 5,205 proceeded to checkout—a step conversion rate of 35.0%, leaving a 65.0% drop-off rate (9,665 sessions).
   - **So what:** Users demonstrate explicit buying intent by adding items to their cart, but friction in the cart experience prevents nearly two-thirds from beginning checkout.

3. **12 product categories underperform the site average listing conversion rate.**
   12 of 23 ranked categories sit below the 19.3% site average seen-to-carted rate with 95% confidence. The top three by carts at stake are Apparel (14.1% seen-to-carted rate, 345 carts at stake, CI [13.2%, 14.9%]), Bags (13.6% seen-to-carted rate, 273 carts at stake, CI [12.7%, 14.6%]), and Sale (17.0% seen-to-carted rate, 228 carts at stake, CI [16.3%, 17.8%]), representing 846 carts at stake together out of 2,204 across all 12 flagged categories.
   - **So what:** Poor listing-level conversion across major catalog categories directly suppresses overall cart additions without affecting neutral categories.

4. **Returning sessions convert at over four times the rate of new sessions.**
   Returning sessions achieve a session-to-purchase rate of 1.7% (CI [1.6%, 1.8%]) compared to 0.4% for new sessions (CI [0.4%, 0.4%]), a ratio of 4.1x (chi-square p < 0.001). However, the average Week-1 return rate sits at 4.2% against an 8.0% planning target assumption.
   - **So what:** Bringing existing visitors back represents a higher-converting pool of traffic, though current week-1 retention trails target planning expectations.

## Hypotheses
1. **[MEDIUM confidence — The initial discovery leak stems from site navigation rather than acquisition traffic quality.**
   - **Evidence:** The session medium spread in product view rates is 8.7pp (ranging from referral at 30.3% to CPC at 21.6%), which sits below the 10pp analyst-set threshold used to indicate a traffic-quality problem, leaning the verdict toward a site-level leak.
   - **Confounder:** Very short, untagged sessions labeled `(not set)` represent 26.2% of total sessions and account for 31.0% of the discovery loss; because these sessions lack tag context, their intent cannot be fully ruled out.

2. **[MEDIUM confidence — Product listing layout and presentation drive underperformance in flagged categories.**
   - **Evidence:** 12 of 23 categories perform below the 19.3% site average seen-to-carted rate with 95% confidence, led by Apparel (14.1%) and Bags (13.6%).
   - **Confounder:** Price and listing composition: median listed price exhibits a weak negative correlation with seen-to-carted rate (-0.33, p = 0.123), meaning cheaper item impulse buying is not ruled out, and categories displayed in broad, mixed lists are seen by less focused sessions than dedicated listings.

3. **[MEDIUM confidence — Increasing cohort return rates will increase overall purchase volume.**
   - **Evidence:** Returning sessions convert at 1.7% compared to 0.4% for first-time sessions (p < 0.001), and reaching the 8.0% planning target would add 9,444 returning sessions generating an estimated $2.6k–$12.0k in revenue (planning assumption estimate).
   - **Confounder:** Self-selection bias: users who return unprompted possess higher inherent buying intent, so re-engaging passive users may not achieve the full 4.1x conversion multiplier of organic returning sessions.

## Recommended Actions
1. **Audit homepage and landing page product navigation.**
   - **What:** Redesign top-level listing access and review CPC landing page layouts to decrease friction between session start and product display lists.
   - **Why:** 78.8% of sessions (279,596 sessions) fail to view products, and the 8.7pp channel spread leans toward a site navigation issue.
   - **Metric:** Lift the overall session-to-product-view conversion rate above the current 21.2%.

2. **Optimize listing views for underperforming catalog categories.**
   - **What:** Re-merchandise and improve listing layout presentations for Apparel, Bags, and Sale.
   - **Why:** These three categories account for 846 carts at stake (Apparel at 14.1%, Bags at 13.6%, Sale at 17.0% seen-to-carted rates versus the 19.3% site average).
   - **Metric:** Increase seen-to-carted rates for Apparel, Bags, and Sale toward the 19.3% site average (planning assumption).

3. **Deploy re-engagement workflows to improve retention.**
   - **What:** Implement post-session email and message touchpoints to encourage week-1 return visits.
   - **Why:** Returning sessions convert at 1.7% versus 0.4% for new sessions, while week-1 return rates currently sit at 4.2%.
   - **Metric:** Increase average Week-1 return rate toward the 8.0% planning target assumption, aiming for 9,444 extra returning sessions.

## What to Measure Next
1. **User-scoped cross-session funnel analysis:** Map user IDs across multiple visits to determine how many of the 2,127 purchasing sessions occurring outside the sequential funnel represent multi-session user journeys where carts were populated in earlier visits.
2. **List position and impression depth tracking:** Evaluate whether the lower seen-to-carted conversion rates in Apparel and Bags are caused by item position depth in search and category list pages.
3. **Short-session landing page technical audit:** Analyze technical load logs and exit URLs specifically for `(not set)` untagged traffic to confirm whether missing medium tags correlate with immediate page bounces or broken tracking scripts.

Brief saved to product_brief_ga4_funnel.md
Context saved to product_brief_context.json
